# **IV. Análisis de variables categóricas (Oncológicos vs Control)**

## Objetivo
Este notebook tiene el fin de analizar la distribución de frecuencias de las variables categóricas presentes en el dataset clasificado del GRD (2019-2024). Para evitar sesgos y permitir comparabilidad, se generarán matrices de frecuencias separadas para la población oncológica, el grupo de control (sin cáncer) y el total del hospital.

## Proceso
1. **Carga Optimizada y Estandarización**: Carga de forma selectiva las columnas de los archivos CSV anuales masivos, estandarizando nombres y filtrando por `CATEGORIA_CANCER`.
2. **Análisis de distribución tripartito**: Genera matrices de frecuencias absolutas y relativas (%) por año, separadas en tres subcarpetas: `Frecuencias (oncológicos)`, `Frecuencias (control)` y `Frecuencias (total)`.
3. **Evaluación de cardinalidad**: Permite justificar el uso, agrupación o descarte de las variables categóricas (como procedencias, previsiones o diagnósticos) para las posteriores etapas de modelado y selección de características.

In [1]:
# ============================================================================
# CONFIGURACIÓN INICIAL: Imports y Rutas
# ============================================================================

import pandas as pd
import os
import warnings 
import unicodedata
import re

# Suprimir advertencias sobre cambios de tipo de dato (dtype)
warnings.simplefilter(action='ignore', category=pd.errors.DtypeWarning)

# Ruta de datos clasificados (que contienen TODA la población y la columna CATEGORIA_CANCER)
carpeta = "../../Datos/Datos clasificados"

# Lista de archivos a procesar (años 2019-2024)
archivos = [
    "GRD_CLASIFICADO_2019.csv",
    "GRD_CLASIFICADO_2020.csv",
    "GRD_CLASIFICADO_2021.csv",
    "GRD_CLASIFICADO_2022.csv",
    "GRD_CLASIFICADO_2023.csv",
    "GRD_CLASIFICADO_2024.csv"
]

# Mapeo de equivalencias para estandarizar nombres de columnas inconsistentes
mapa_columnas = {
    "ï»¿COD_HOSPITAL": "COD_HOSPITAL",   # Corrige corrupción de encoding (BOM UTF-8)
    "ID_BENEFICIARIO": "CIP_ENCRIPTADO"  # Renombra a estándar de proyecto
}

In [2]:
# ============================================================================
# UTILIDADES: Funciones de Limpieza y Normalización
# ============================================================================

def normalizar_texto(texto):
    if texto is None:
        return None
    try:
        texto = texto.encode("latin1").decode("utf-8")
    except:
        pass
    texto = unicodedata.normalize("NFKD", texto)
    texto = "".join(c for c in texto if not unicodedata.combining(c))
    return texto

def obtener_subcarpeta(columna):
    if re.match(r"^SERVICIOTRASLADO\d+$", columna):
        return "servicios_traslado"
    if re.match(r"^(CONDICIONDEALTANEONATO\d+|PESORN\d+|SEXORN\d+|RN\d+ESTADO)$", columna):
        return "neonatal"
    if re.match(r"^DIAGNOSTICO\d+$", columna):
        return "diagnosticos"
    if re.match(r"^PROCEDIMIENTO\d+$", columna):
        return "procedimientos"
    return ""

def homologar_categorias(col, nombre_columna):
    if nombre_columna == "ETNIA":
        mapa = {
            "NINGUNA": "NINGUNO",
            "NINGUN": "NINGUNO",
            "SIN INFORMACION": "NINGUNO",
            "NO APLICA": "NINGUNO"
        }
        col = col.replace(mapa)
    return col

# ============================================================================
# FUNCIÓN PRINCIPAL: Cálculo de Distribución Simultánea (Ultra-optimizada)
# ============================================================================

def distribucion_categorica_matriz(carpeta, archivos, mapa_columnas, columna):
    """
    - Descripción: Lee los archivos una sola vez, separa las cohortes en RAM y 
                   genera simultáneamente las matrices de frecuencias para: 
                   Total, Oncológicos y Control.
    - Entradas:
        - columna (str): Nombre de la variable categórica a analizar.
    - Salidas:
        - tuple: (matriz_total, matriz_onco, matriz_control)
    """
    print(f"\n[{columna}] Extrayendo y procesando datos masivos (Una sola pasada)...")
    
    # Listas para acumular los conteos de todos los años
    resultados_total = []
    resultados_onco = []
    resultados_control = []
    
    categorias_globales = set()

    for archivo in archivos:
        ruta = os.path.join(carpeta, archivo)
        año_archivo = int(archivo[-8:-4])

        # 1. Verificar si la columna existe antes de cargar datos
        encabezados = pd.read_csv(ruta, nrows=0).rename(columns=mapa_columnas).columns
        if columna not in encabezados:
            continue

        # 2. Leer SOLO las columnas necesarias
        columnas_a_leer = [col for col in pd.read_csv(ruta, nrows=0).columns 
                           if col == columna or col == 'CATEGORIA_CANCER' or mapa_columnas.get(col) == columna]
        
        df = pd.read_csv(ruta, usecols=columnas_a_leer, low_memory=False).rename(columns=mapa_columnas)

        if df.empty:
            continue

        # 3. Limpieza y estandarización global (para no repetir el trabajo)
        col_original = df[columna]
        mask_null_real = col_original.isna()
        col = col_original.copy().astype("object")

        col[~mask_null_real] = (
            col[~mask_null_real]
            .astype(str)
            .str.strip()
            .apply(normalizar_texto)
            .str.upper()
        )

        col = homologar_categorias(col, columna)
        col = col.astype("object")
        col = col.mask(col.isin(["", "NAN", "NONE", "NULL"]), "NULOS")
        col = col.astype("string")
        col[mask_null_real] = "NULOS"
        
        # Asignar la columna limpia de vuelta al DataFrame
        df[columna] = col

        # 4. Separación de cohortes en memoria RAM (Operación Vectorizada)
        mask_control = df['CATEGORIA_CANCER'] == 'SIN_CANCER'
        
        df_control = df[mask_control]
        df_onco = df[~mask_control]
        
        # 5. Conteos Simultáneos
        conteo_total = df[columna].value_counts()
        conteo_onco = df_onco[columna].value_counts()
        conteo_control = df_control[columna].value_counts()

        # Actualizar conjunto global de categorías
        categorias_globales.update(conteo_total.index.tolist())

        # 6. Almacenar resultados para cada grupo
        for cat, freq in conteo_total.items():
            resultados_total.append({"Categoría": cat, "Año": año_archivo, "Frecuencia": freq})
            
        for cat, freq in conteo_onco.items():
            resultados_onco.append({"Categoría": cat, "Año": año_archivo, "Frecuencia": freq})
            
        for cat, freq in conteo_control.items():
            resultados_control.append({"Categoría": cat, "Año": año_archivo, "Frecuencia": freq})

    if not resultados_total:
        print(f"  -> No se encontraron datos para {columna}.")
        return None, None, None

    # =====================================================================
    # FUNCIÓN AUXILIAR INTERNA: Construir Matriz Pivote
    # =====================================================================
    def construir_matriz(lista_resultados):
        if not lista_resultados:
            return None
        df_res = pd.DataFrame(lista_resultados)
        matriz = df_res.pivot_table(index="Categoría", columns="Año", values="Frecuencia", fill_value=0)
        matriz = matriz.sort_index(axis=1)
        matriz["TOTAL"] = matriz.sum(axis=1)
        matriz = matriz.sort_values("TOTAL", ascending=False).drop(columns="TOTAL")
        matriz = matriz.fillna(0).astype(int)

        matriz_pct = matriz.div(matriz.sum(axis=0), axis=1) * 100
        matriz_final = matriz.copy().astype(str)
        for col_año in matriz.columns:
            matriz_final[col_año] = (
                matriz[col_año].astype(int).astype(str)
                + " (" + matriz_pct[col_año].round(1).astype(str) + "%)"
            )
        matriz_final.columns.name = None
        matriz_final.index.name = None
        return matriz_final

    # 7. Construir las tres matrices simultáneamente
    matriz_total = construir_matriz(resultados_total)
    matriz_onco = construir_matriz(resultados_onco)
    matriz_control = construir_matriz(resultados_control)

    # 8. Reporte de Cardinalidad
    categorias_limpias = sorted([c for c in categorias_globales if c not in ["NULOS", "DESCONOCIDO"]])
    print(f"✓ Cardinalidad {columna}: {len(categorias_limpias)} categorías únicas")

    # 9. Guardar en disco (Creación de subcarpetas automáticas)
    subcarpeta = obtener_subcarpeta(columna)
    nombres_y_matrices = [
        ("total", matriz_total),
        ("oncológicos", matriz_onco),
        ("control", matriz_control)
    ]
    
    for nombre_pob, matriz in nombres_y_matrices:
        if matriz is not None:
            carpeta_salida = os.path.join(f"../../Resultados/Resultados (etapa 1 y 2)/Frecuencias (categóricas)/Frecuencias ({nombre_pob})", subcarpeta)
            os.makedirs(carpeta_salida, exist_ok=True)
            ruta_salida = os.path.join(carpeta_salida, f"{columna.lower()}_frecuencia.csv")
            matriz.to_csv(ruta_salida, encoding="utf-8-sig")

    print(f"✓ Archivos CSV generados y guardados en las 3 carpetas correspondientes.")
    
    # 10. Retornar la tupla con las 3 tablas para visualización
    return matriz_total, matriz_onco, matriz_control

In [3]:
def mostrar_resultados(matriz_total, matriz_onco, matriz_control, variable):
    mensaje_total = f"1. Frecuencia para cohorte total ({variable}):"
    mensaje_oncologico = f"2. Frecuencia para cohorte oncológica ({variable}):"
    mensaje_control = f"3. Frecuencia para cohorte control ({variable}):"
    display(mensaje_total)
    display(matriz_total)
    display(mensaje_oncologico)
    display(matriz_onco)
    display(mensaje_control)
    display(matriz_control)

## **1. Variables categóricas objetivo:**

### **1. Mortalidad (TIPOALTA)**
- **Tipo de variable**: Objetivo (*Target*).
- **Descripción**: Condición de egreso o destino del paciente al finalizar su hospitalización.
- **Veredicto**: **Retener y Binarizar**. Esta variable representa la Verdad Fundamental (*Ground Truth*) del desenlace clínico. Al cruzar los datos, se observa un contraste estadístico crítico: la cohorte oncológica presenta una tasa de mortalidad de entre 5.4% y 6.9%, casi el triple que la del grupo control (~2.3%). Para habilitar los algoritmos de selección de características, las 11 categorías originales se binarizarán en una variable dicotómica: `FALLECIDO` (Clase Positiva) vs. `VIVO` (agrupando Alta voluntaria, Domicilio, Derivaciones, etc.). Las categorías marginales sin información (`NO IDENTIFICADA`, `DESCONOCIDO`) serán imputadas a la moda o descartadas.

In [4]:
total, oncologico, control = distribucion_categorica_matriz(carpeta, archivos, mapa_columnas,"TIPOALTA")
mostrar_resultados(total, oncologico, control, "TIPOALTA")


[TIPOALTA] Extrayendo y procesando datos masivos (Una sola pasada)...
✓ Cardinalidad TIPOALTA: 11 categorías únicas
✓ Archivos CSV generados y guardados en las 3 carpetas correspondientes.


'1. Frecuencia para cohorte total (TIPOALTA):'

,2019,2020,2021,2022,2023,2024
DOMICILIO,1052020 (91.4%),694899 (88.9%),716270 (87.7%),838234 (89.9%),936205 (90.1%),967852 (89.1%)
FALLECIDO,29584 (2.6%),29942 (3.8%),31965 (3.9%),27379 (2.9%),25136 (2.4%),26682 (2.5%)
HOSPITALIZACION DOMICILIARIA,12987 (1.1%),14071 (1.8%),22403 (2.7%),23408 (2.5%),29298 (2.8%),36732 (3.4%)
DERIVACION OTRO HOSPITAL DEL SERVICIO,25206 (2.2%),17839 (2.3%),18321 (2.2%),16488 (1.8%),20505 (2.0%),24923 (2.3%)
ALTA VOLUNTARIA,9963 (0.9%),7569 (1.0%),9059 (1.1%),10721 (1.1%),10317 (1.0%),10651 (1.0%)
DERIVACION OTRO HOSPITAL DE LA RED NACIONAL,8415 (0.7%),6871 (0.9%),7194 (0.9%),6539 (0.7%),7472 (0.7%),7938 (0.7%)
"DERIVACION A OTROS CENTROS (CARCEL, HOGAR DE",2980 (0.3%),4396 (0.6%),4038 (0.5%),3385 (0.4%),3497 (0.3%),3745 (0.3%)
FUGA DEL PACIENTE,4141 (0.4%),2719 (0.3%),2593 (0.3%),3184 (0.3%),3224 (0.3%),3299 (0.3%)
DERIVACION INST. PRIVADA (COMPRA DE SERVICIOS,4411 (0.4%),2430 (0.3%),3688 (0.5%),2373 (0.3%),2886 (0.3%),2865 (0.3%)
DERIVACION INST. PRIVADA (VOLUNTARIO),1614 (0.1%),1152 (0.1%),1378 (0.2%),1126 (0.1%),1047 (0.1%),1126 (0.1%)


'2. Frecuencia para cohorte oncológica (TIPOALTA):'

,2019,2020,2021,2022,2023,2024
DOMICILIO,88008 (90.0%),56334 (86.8%),58867 (86.7%),67703 (86.7%),78385 (86.4%),84680 (85.4%)
FALLECIDO,5321 (5.4%),4462 (6.9%),4423 (6.5%),4850 (6.2%),5314 (5.9%),6009 (6.1%)
HOSPITALIZACION DOMICILIARIA,1319 (1.3%),1471 (2.3%),2110 (3.1%),2741 (3.5%),3531 (3.9%),4473 (4.5%)
DERIVACION OTRO HOSPITAL DEL SERVICIO,1631 (1.7%),1198 (1.8%),1069 (1.6%),1242 (1.6%),1671 (1.8%),2147 (2.2%)
DERIVACION OTRO HOSPITAL DE LA RED NACIONAL,659 (0.7%),584 (0.9%),541 (0.8%),560 (0.7%),688 (0.8%),755 (0.8%)
ALTA VOLUNTARIA,526 (0.5%),498 (0.8%),569 (0.8%),646 (0.8%),710 (0.8%),652 (0.7%)
"DERIVACION A OTROS CENTROS (CARCEL, HOGAR DE",76 (0.1%),171 (0.3%),135 (0.2%),116 (0.1%),143 (0.2%),139 (0.1%)
DERIVACION INST. PRIVADA (COMPRA DE SERVICIOS,144 (0.1%),97 (0.1%),112 (0.2%),91 (0.1%),142 (0.2%),167 (0.2%)
FUGA DEL PACIENTE,75 (0.1%),60 (0.1%),49 (0.1%),69 (0.1%),77 (0.1%),79 (0.1%)
DERIVACION INST. PRIVADA (VOLUNTARIO),65 (0.1%),50 (0.1%),51 (0.1%),51 (0.1%),54 (0.1%),52 (0.1%)


'3. Frecuencia para cohorte control (TIPOALTA):'

,2019,2020,2021,2022,2023,2024
DOMICILIO,964012 (91.5%),638565 (89.1%),657403 (87.8%),770531 (90.1%),857820 (90.4%),883172 (89.5%)
FALLECIDO,24263 (2.3%),25480 (3.6%),27542 (3.7%),22529 (2.6%),19822 (2.1%),20673 (2.1%)
HOSPITALIZACION DOMICILIARIA,11668 (1.1%),12600 (1.8%),20293 (2.7%),20667 (2.4%),25767 (2.7%),32259 (3.3%)
DERIVACION OTRO HOSPITAL DEL SERVICIO,23575 (2.2%),16641 (2.3%),17252 (2.3%),15246 (1.8%),18834 (2.0%),22776 (2.3%)
ALTA VOLUNTARIA,9437 (0.9%),7071 (1.0%),8490 (1.1%),10075 (1.2%),9607 (1.0%),9999 (1.0%)
DERIVACION OTRO HOSPITAL DE LA RED NACIONAL,7756 (0.7%),6287 (0.9%),6653 (0.9%),5979 (0.7%),6784 (0.7%),7183 (0.7%)
"DERIVACION A OTROS CENTROS (CARCEL, HOGAR DE",2904 (0.3%),4225 (0.6%),3903 (0.5%),3269 (0.4%),3354 (0.4%),3606 (0.4%)
FUGA DEL PACIENTE,4066 (0.4%),2659 (0.4%),2544 (0.3%),3115 (0.4%),3147 (0.3%),3220 (0.3%)
DERIVACION INST. PRIVADA (COMPRA DE SERVICIOS,4267 (0.4%),2333 (0.3%),3576 (0.5%),2282 (0.3%),2744 (0.3%),2698 (0.3%)
DERIVACION INST. PRIVADA (VOLUNTARIO),1549 (0.1%),1102 (0.2%),1327 (0.2%),1075 (0.1%),993 (0.1%),1074 (0.1%)


### **2. Severidad (SEVERIDAD)**
- **Tipo de variable**: Objetivo (*Target*).
- **Descripción**: Métrica oficial del agrupador GRD que clasifica la extensión de la descompensación fisiológica y pérdida de función de órganos.
- **Veredicto**: **Retener como variable ordinal**. Es el segundo objetivo de caracterización del estudio. La matriz de frecuencias justifica plenamente su estudio: más del 29% de los pacientes oncológicos alcanzan la categoría de Gravedad Mayor (Nivel 3), en contraste con el ~19% del grupo control. Se mantendrá su escala discreta (0 a 3) nativa.

| ID | Gravedad     |
|----|--------------|
| 0  | Sin gravedad |
| 1  | Menor        |
| 2  | Moderada     |
| 3  | Mayor        |

In [5]:
total, oncologico, control = distribucion_categorica_matriz(carpeta, archivos, mapa_columnas,"IR_29301_SEVERIDAD")
mostrar_resultados(total, oncologico, control, "IR_29301_SEVERIDAD")


[IR_29301_SEVERIDAD] Extrayendo y procesando datos masivos (Una sola pasada)...
✓ Cardinalidad IR_29301_SEVERIDAD: 4 categorías únicas
✓ Archivos CSV generados y guardados en las 3 carpetas correspondientes.


'1. Frecuencia para cohorte total (IR_29301_SEVERIDAD):'

,2019,2020,2021,2022,2023,2024
1,558571 (48.5%),366293 (46.8%),340136 (41.6%),373440 (40.0%),388694 (37.4%),387258 (35.7%)
2,242291 (21.0%),176554 (22.6%),183822 (22.5%),213260 (22.9%),253086 (24.3%),268865 (24.8%)
3,147998 (12.9%),159400 (20.4%),190013 (23.3%),191342 (20.5%),204227 (19.6%),217624 (20.0%)
0,202517 (17.6%),79664 (10.2%),102938 (12.6%),154772 (16.6%),193550 (18.6%),212051 (19.5%)
DESCONOCIDO,21 (0.0%),0 (0.0%),0 (0.0%),23 (0.0%),30 (0.0%),15 (0.0%)


'2. Frecuencia para cohorte oncológica (IR_29301_SEVERIDAD):'

,2019,2020,2021,2022,2023,2024
2,30680 (31.4%),23807 (36.7%),23718 (34.9%),26710 (34.2%),31917 (35.2%),34607 (34.9%)
3,19771 (20.2%),18413 (28.4%),19393 (28.6%),22960 (29.4%),26813 (29.6%),30265 (30.5%)
1,26703 (27.3%),19058 (29.4%),19126 (28.2%),20488 (26.2%),21770 (24.0%),23198 (23.4%)
0,20675 (21.1%),3648 (5.6%),5689 (8.4%),7909 (10.1%),10215 (11.3%),11081 (11.2%)
DESCONOCIDO,0 (0.0%),0 (0.0%),0 (0.0%),2 (0.0%),0 (0.0%),2 (0.0%)


'3. Frecuencia para cohorte control (IR_29301_SEVERIDAD):'

,2019,2020,2021,2022,2023,2024
1,531868 (50.5%),347235 (48.4%),321010 (42.9%),352952 (41.3%),366924 (38.7%),364060 (36.9%)
2,211611 (20.1%),152747 (21.3%),160104 (21.4%),186550 (21.8%),221169 (23.3%),234258 (23.7%)
3,128227 (12.2%),140987 (19.7%),170620 (22.8%),168382 (19.7%),177414 (18.7%),187359 (19.0%)
0,181842 (17.3%),76016 (10.6%),97249 (13.0%),146863 (17.2%),183335 (19.3%),200970 (20.4%)
DESCONOCIDO,21 (0.0%),0 (0.0%),0 (0.0%),21 (0.0%),30 (0.0%),13 (0.0%)


## **2. Resto de variables categóricas:**

### **3. Sexo (SEXO)**
- **Tipo de variable**: Demográfica.
- **Descripción**: Sexo biológico del paciente.
- **Cardinalidad**: Tiene 2 categorías y la etiqueta "DESCONOCIDO" es marginal (0.0%).
- **Veredicto**: **Retener**. Presenta una cardinalidad perfecta (2 categorías útiles) y una distribución equilibrada y constante a lo largo de los 6 años (~59% mujeres, ~41% hombres en la población general; y ~53% mujeres, ~46% hombres en oncología). Es un factor demográfico fundamental en la literatura médica para explicar la prevalencia y severidad de perfiles clínicos.

In [6]:
total, oncologico, control = distribucion_categorica_matriz(carpeta, archivos, mapa_columnas,"SEXO")
mostrar_resultados(total, oncologico, control, "SEXO")


[SEXO] Extrayendo y procesando datos masivos (Una sola pasada)...
✓ Cardinalidad SEXO: 2 categorías únicas
✓ Archivos CSV generados y guardados en las 3 carpetas correspondientes.


'1. Frecuencia para cohorte total (SEXO):'

,2019,2020,2021,2022,2023,2024
MUJER,681573 (59.2%),461635 (59.0%),475306 (58.2%),553565 (59.3%),612203 (58.9%),632623 (58.3%)
HOMBRE,469670 (40.8%),320142 (40.9%),341540 (41.8%),379220 (40.7%),427345 (41.1%),453074 (41.7%)
DESCONOCIDO,155 (0.0%),134 (0.0%),63 (0.0%),52 (0.0%),39 (0.0%),116 (0.0%)


'2. Frecuencia para cohorte oncológica (SEXO):'

,2019,2020,2021,2022,2023,2024
MUJER,53312 (54.5%),34474 (53.1%),36044 (53.1%),41965 (53.8%),48594 (53.6%),53224 (53.7%)
HOMBRE,44513 (45.5%),30448 (46.9%),31882 (46.9%),36101 (46.2%),42120 (46.4%),45910 (46.3%)
DESCONOCIDO,4 (0.0%),4 (0.0%),0 (0.0%),3 (0.0%),1 (0.0%),19 (0.0%)


'3. Frecuencia para cohorte control (SEXO):'

,2019,2020,2021,2022,2023,2024
MUJER,628261 (59.6%),427161 (59.6%),439262 (58.6%),511600 (59.9%),563609 (59.4%),579399 (58.7%)
HOMBRE,425157 (40.4%),289694 (40.4%),309658 (41.3%),343119 (40.1%),385225 (40.6%),407164 (41.3%)
DESCONOCIDO,151 (0.0%),130 (0.0%),63 (0.0%),49 (0.0%),38 (0.0%),97 (0.0%)


### **4. Etnia (ETNIA)**
- **Tipo de variable**: Demográfica.
- **Descripción**: Pertenencia declarada a algún pueblo originario.
- **Cardinalidad**: Tiene 12 categorías. 
- **Veredicto**: Transformar y Binarizar (ES_PUEBLO_ORIGINARIO). El análisis revela una varianza nativa casi nula y un quiebre en las políticas de registro del MINSAL/FONASA (en 2022 y 2023 la categoría NINGUNO desaparece y es reemplazada íntegramente por OTRO). Para rescatar el valor epidemiológico de esta variable como determinante social de la salud, se aplicará ingeniería de características creando una variable dicotómica. Todas las categorías específicas (Mapuche, Aymara, Rapa Nui, etc.) conformarán la clase positiva (1), mientras que NINGUNO y OTRO conformarán la clase de control (0). Esta transformación neutraliza el error de codificación interanual y previene la alta dimensionalidad, permitiendo que las técnicas de Bootstrapping evalúen su impacto real en los perfiles de severidad y consumo.

In [7]:
total, oncologico, control = distribucion_categorica_matriz(carpeta, archivos, mapa_columnas,"ETNIA")
mostrar_resultados(total, oncologico, control, "ETNIA")


[ETNIA] Extrayendo y procesando datos masivos (Una sola pasada)...
✓ Cardinalidad ETNIA: 12 categorías únicas
✓ Archivos CSV generados y guardados en las 3 carpetas correspondientes.


'1. Frecuencia para cohorte total (ETNIA):'

,2019,2020,2021,2022,2023,2024
NINGUNO,816886 (70.9%),668294 (85.5%),769595 (94.2%),0 (0.0%),0 (0.0%),1058981 (97.5%)
OTRO,313517 (27.2%),96019 (12.3%),28492 (3.5%),912573 (97.8%),1016554 (97.8%),0 (0.0%)
MAPUCHE,19224 (1.7%),14976 (1.9%),14867 (1.8%),15336 (1.6%),17334 (1.7%),20494 (1.9%)
AYMARA,659 (0.1%),1249 (0.2%),1968 (0.2%),2247 (0.2%),2355 (0.2%),2637 (0.2%)
RAPA NUI (PASCUENSE),41 (0.0%),24 (0.0%),124 (0.0%),1162 (0.1%),1389 (0.1%),1339 (0.1%)
DIAGUITA,94 (0.0%),98 (0.0%),392 (0.0%),682 (0.1%),837 (0.1%),1106 (0.1%)
KAWESQAR,689 (0.1%),820 (0.1%),835 (0.1%),77 (0.0%),201 (0.0%),250 (0.0%)
QUECHUA,180 (0.0%),217 (0.0%),289 (0.0%),399 (0.0%),407 (0.0%),403 (0.0%)
YAGAN (YAMANA),25 (0.0%),114 (0.0%),215 (0.0%),211 (0.0%),247 (0.0%),276 (0.0%)
COLLA,45 (0.0%),64 (0.0%),84 (0.0%),75 (0.0%),81 (0.0%),169 (0.0%)


'2. Frecuencia para cohorte oncológica (ETNIA):'

,2019,2020,2021,2022,2023,2024
NINGUNO,73566 (75.2%),56185 (86.5%),64882 (95.5%),0 (0.0%),0 (0.0%),96967 (97.8%)
OTRO,22863 (23.4%),7361 (11.3%),1766 (2.6%),76762 (98.3%),88964 (98.1%),0 (0.0%)
MAPUCHE,1332 (1.4%),1224 (1.9%),1095 (1.6%),1007 (1.3%),1355 (1.5%),1771 (1.8%)
AYMARA,9 (0.0%),51 (0.1%),66 (0.1%),91 (0.1%),143 (0.2%),139 (0.1%)
RAPA NUI (PASCUENSE),4 (0.0%),0 (0.0%),17 (0.0%),123 (0.2%),115 (0.1%),141 (0.1%)
KAWESQAR,42 (0.0%),63 (0.1%),50 (0.1%),6 (0.0%),6 (0.0%),15 (0.0%)
DIAGUITA,2 (0.0%),4 (0.0%),16 (0.0%),30 (0.0%),58 (0.1%),69 (0.1%)
YAGAN (YAMANA),0 (0.0%),25 (0.0%),28 (0.0%),17 (0.0%),30 (0.0%),21 (0.0%)
QUECHUA,10 (0.0%),7 (0.0%),3 (0.0%),18 (0.0%),9 (0.0%),11 (0.0%)
LICAN ANTAI (ATACAMENO),0 (0.0%),6 (0.0%),0 (0.0%),8 (0.0%),25 (0.0%),8 (0.0%)


'3. Frecuencia para cohorte control (ETNIA):'

,2019,2020,2021,2022,2023,2024
NINGUNO,743320 (70.6%),612109 (85.4%),704713 (94.1%),0 (0.0%),0 (0.0%),962014 (97.5%)
OTRO,290654 (27.6%),88658 (12.4%),26726 (3.6%),835811 (97.8%),927590 (97.8%),0 (0.0%)
MAPUCHE,17892 (1.7%),13752 (1.9%),13772 (1.8%),14329 (1.7%),15979 (1.7%),18723 (1.9%)
AYMARA,650 (0.1%),1198 (0.2%),1902 (0.3%),2156 (0.3%),2212 (0.2%),2498 (0.3%)
RAPA NUI (PASCUENSE),37 (0.0%),24 (0.0%),107 (0.0%),1039 (0.1%),1274 (0.1%),1198 (0.1%)
DIAGUITA,92 (0.0%),94 (0.0%),376 (0.1%),652 (0.1%),779 (0.1%),1037 (0.1%)
KAWESQAR,647 (0.1%),757 (0.1%),785 (0.1%),71 (0.0%),195 (0.0%),235 (0.0%)
QUECHUA,170 (0.0%),210 (0.0%),286 (0.0%),381 (0.0%),398 (0.0%),392 (0.0%)
YAGAN (YAMANA),25 (0.0%),89 (0.0%),187 (0.0%),194 (0.0%),217 (0.0%),255 (0.0%)
COLLA,44 (0.0%),64 (0.0%),81 (0.0%),68 (0.0%),71 (0.0%),158 (0.0%)


### **5. Provincia de Residencia (PROVINCIA)**
- **Tipo de variable**: Geográfica / Demográfica.
- **Descripción**: Provincia registrada como domicilio del paciente.
- **Cardinalidad**: Tiene 57 categorías (una cantidad grande)
- **Veredicto**: **Retener y Transformar (Agrupar a Nivel Regional)**. Mantener las 57 categorías provinciales generaría una matriz excesivamente dispersa. Sin embargo, el dato geográfico es valioso. La estrategia metodológica será mapear y agrupar las 57 provincias en sus respectivas 16 Regiones de Chile. Esto reduce drásticamente la cardinalidad, mantiene la explicabilidad territorial y facilita la convergencia de los algoritmos de aprendizaje automático al evaluar el traslado y centralización de los pacientes.

In [8]:
total, oncologico, control = distribucion_categorica_matriz(carpeta, archivos, mapa_columnas,"PROVINCIA")
mostrar_resultados(total, oncologico, control, "PROVINCIA")


[PROVINCIA] Extrayendo y procesando datos masivos (Una sola pasada)...
✓ Cardinalidad PROVINCIA: 57 categorías únicas
✓ Archivos CSV generados y guardados en las 3 carpetas correspondientes.


'1. Frecuencia para cohorte total (PROVINCIA):'

,2019,2020,2021,2022,2023,2024
SANTIAGO,290205 (25.2%),204076 (26.1%),205803 (25.2%),232534 (24.9%),248038 (23.9%),248393 (22.9%)
CONCEPCION,66585 (5.8%),43516 (5.6%),47713 (5.8%),55095 (5.9%),69925 (6.7%),77672 (7.2%)
CAUTIN,57863 (5.0%),40466 (5.2%),38557 (4.7%),41622 (4.5%),47168 (4.5%),59183 (5.5%)
CORDILLERA,48166 (4.2%),32440 (4.1%),37480 (4.6%),40271 (4.3%),41957 (4.0%),42239 (3.9%)
VALPARAISO,63552 (5.5%),28405 (3.6%),28632 (3.5%),33483 (3.6%),42867 (4.1%),44310 (4.1%)
ELQUI,34870 (3.0%),24793 (3.2%),27168 (3.3%),29738 (3.2%),36556 (3.5%),37796 (3.5%)
BIO-BIO,33956 (2.9%),22459 (2.9%),23155 (2.8%),29483 (3.2%),31693 (3.0%),31095 (2.9%)
CACHAPOAL,31129 (2.7%),21896 (2.8%),22167 (2.7%),24949 (2.7%),25681 (2.5%),26974 (2.5%)
TALCA,27068 (2.4%),20942 (2.7%),21016 (2.6%),24653 (2.6%),26197 (2.5%),30924 (2.8%)
LLANQUIHUE,28862 (2.5%),20452 (2.6%),21484 (2.6%),23364 (2.5%),27543 (2.6%),28922 (2.7%)


'2. Frecuencia para cohorte oncológica (PROVINCIA):'

,2019,2020,2021,2022,2023,2024
SANTIAGO,24391 (24.9%),16424 (25.3%),17256 (25.4%),19849 (25.4%),22671 (25.0%),23842 (24.0%)
CONCEPCION,6208 (6.3%),4570 (7.0%),5221 (7.7%),6136 (7.9%),7384 (8.1%),8217 (8.3%)
CORDILLERA,6913 (7.1%),3429 (5.3%),4048 (6.0%),4528 (5.8%),4741 (5.2%),5031 (5.1%)
VALPARAISO,8147 (8.3%),3175 (4.9%),2695 (4.0%),3543 (4.5%),4945 (5.5%),5115 (5.2%)
CAUTIN,4480 (4.6%),3484 (5.4%),3243 (4.8%),3356 (4.3%),4148 (4.6%),5341 (5.4%)
ELQUI,2316 (2.4%),1892 (2.9%),2087 (3.1%),2352 (3.0%),2998 (3.3%),3301 (3.3%)
TALCA,2079 (2.1%),1743 (2.7%),1928 (2.8%),2047 (2.6%),2282 (2.5%),2564 (2.6%)
BIO-BIO,2125 (2.2%),1518 (2.3%),1661 (2.4%),2280 (2.9%),2367 (2.6%),2610 (2.6%)
CACHAPOAL,2041 (2.1%),1834 (2.8%),1880 (2.8%),2086 (2.7%),2214 (2.4%),2332 (2.4%)
LLANQUIHUE,2360 (2.4%),1802 (2.8%),1750 (2.6%),1679 (2.2%),2182 (2.4%),2346 (2.4%)


'3. Frecuencia para cohorte control (PROVINCIA):'

,2019,2020,2021,2022,2023,2024
SANTIAGO,265814 (25.2%),187652 (26.2%),188547 (25.2%),212685 (24.9%),225367 (23.8%),224551 (22.8%)
CONCEPCION,60377 (5.7%),38946 (5.4%),42492 (5.7%),48959 (5.7%),62541 (6.6%),69455 (7.0%)
CAUTIN,53383 (5.1%),36982 (5.2%),35314 (4.7%),38266 (4.5%),43020 (4.5%),53842 (5.5%)
CORDILLERA,41253 (3.9%),29011 (4.0%),33432 (4.5%),35743 (4.2%),37216 (3.9%),37208 (3.8%)
VALPARAISO,55405 (5.3%),25230 (3.5%),25937 (3.5%),29940 (3.5%),37922 (4.0%),39195 (4.0%)
ELQUI,32554 (3.1%),22901 (3.2%),25081 (3.3%),27386 (3.2%),33558 (3.5%),34495 (3.5%)
BIO-BIO,31831 (3.0%),20941 (2.9%),21494 (2.9%),27203 (3.2%),29326 (3.1%),28485 (2.9%)
CACHAPOAL,29088 (2.8%),20062 (2.8%),20287 (2.7%),22863 (2.7%),23467 (2.5%),24642 (2.5%)
LLANQUIHUE,26502 (2.5%),18650 (2.6%),19734 (2.6%),21685 (2.5%),25361 (2.7%),26576 (2.7%)
TALCA,24989 (2.4%),19199 (2.7%),19088 (2.5%),22606 (2.6%),23915 (2.5%),28360 (2.9%)


### **6. Comuna de Residencia (COMUNA)**
- **Tipo de variable**: Demográfica / Geográfica.
- **Descripción**: Comuna específica donde el paciente tiene registrado su domicilio.
- **Cardinalidad**: Muy Alta (346 categorías únicas).
- **Veredicto**: **Descartar**. La alta fragmentación de la variable genera una matriz excesivamente dispersa (la comuna más frecuente, Puente Alto, apenas representa el ~4% del volumen total, dejando cientos de categorías con frecuencias cercanas a cero). Aplicar *One-Hot Encoding* inyectaría 346 nuevas columnas al modelo, diluyendo el peso de variables clínicas críticas e incrementando el riesgo de sobreajuste. Dado que la distribución territorial y el impacto del traslado ya están contenidos en variables más robustas y agrupadas como `PROVINCIA` y `SERVICIO_SALUD`, la eliminación de la comuna no sacrifica información espacial relevante a nivel macro.

In [9]:
total, oncologico, control = distribucion_categorica_matriz(carpeta, archivos, mapa_columnas,"COMUNA")
mostrar_resultados(total, oncologico, control, "COMUNA")


[COMUNA] Extrayendo y procesando datos masivos (Una sola pasada)...
✓ Cardinalidad COMUNA: 346 categorías únicas
✓ Archivos CSV generados y guardados en las 3 carpetas correspondientes.


'1. Frecuencia para cohorte total (COMUNA):'

,2019,2020,2021,2022,2023,2024
PUENTE ALTO,45411 (3.9%),30473 (3.9%),35432 (4.3%),38233 (4.1%),39464 (3.8%),39492 (3.6%)
LA FLORIDA,23608 (2.1%),17332 (2.2%),19817 (2.4%),22517 (2.4%),23926 (2.3%),21818 (2.0%)
MAIPU,27907 (2.4%),18201 (2.3%),18156 (2.2%),19694 (2.1%),21427 (2.1%),21493 (2.0%)
VALPARAISO,36422 (3.2%),13957 (1.8%),15093 (1.8%),17299 (1.9%),20424 (2.0%),20756 (1.9%)
ARICA,20451 (1.8%),15556 (2.0%),17103 (2.1%),18508 (2.0%),19968 (1.9%),20352 (1.9%)
...,...,...,...,...,...,...
GENERAL LAGOS,1 (0.0%),3 (0.0%),1 (0.0%),0 (0.0%),8 (0.0%),4 (0.0%)
PUTRE,3 (0.0%),0 (0.0%),2 (0.0%),2 (0.0%),5 (0.0%),4 (0.0%)
TORRES DEL PAINE,7 (0.0%),1 (0.0%),1 (0.0%),0 (0.0%),5 (0.0%),0 (0.0%)
CAMARONES,0 (0.0%),1 (0.0%),2 (0.0%),0 (0.0%),2 (0.0%),9 (0.0%)


'2. Frecuencia para cohorte oncológica (COMUNA):'

,2019,2020,2021,2022,2023,2024
PUENTE ALTO,6491 (6.6%),3095 (4.8%),3738 (5.5%),4277 (5.5%),4447 (4.9%),4715 (4.8%)
VALPARAISO,4828 (4.9%),1810 (2.8%),1582 (2.3%),1969 (2.5%),2471 (2.7%),2569 (2.6%)
LA FLORIDA,3056 (3.1%),1657 (2.6%),2029 (3.0%),2008 (2.6%),2253 (2.5%),2154 (2.2%)
MAIPU,2597 (2.7%),1859 (2.9%),1491 (2.2%),1960 (2.5%),2350 (2.6%),2445 (2.5%)
ANTOFAGASTA,2210 (2.3%),1612 (2.5%),1491 (2.2%),1648 (2.1%),1910 (2.1%),2179 (2.2%)
...,...,...,...,...,...,...
LAGUNA BLANCA,0 (0.0%),1 (0.0%),3 (0.0%),0 (0.0%),0 (0.0%),0 (0.0%)
GENERAL LAGOS,1 (0.0%),0 (0.0%),0 (0.0%),0 (0.0%),1 (0.0%),1 (0.0%)
RIO VERDE,0 (0.0%),1 (0.0%),0 (0.0%),0 (0.0%),0 (0.0%),0 (0.0%)
OLLAGUE,0 (0.0%),0 (0.0%),1 (0.0%),0 (0.0%),0 (0.0%),0 (0.0%)


'3. Frecuencia para cohorte control (COMUNA):'

,2019,2020,2021,2022,2023,2024
PUENTE ALTO,38920 (3.7%),27378 (3.8%),31694 (4.2%),33956 (4.0%),35017 (3.7%),34777 (3.5%)
LA FLORIDA,20552 (2.0%),15675 (2.2%),17788 (2.4%),20509 (2.4%),21673 (2.3%),19664 (2.0%)
MAIPU,25310 (2.4%),16342 (2.3%),16665 (2.2%),17734 (2.1%),19077 (2.0%),19048 (1.9%)
VALPARAISO,31594 (3.0%),12147 (1.7%),13511 (1.8%),15330 (1.8%),17953 (1.9%),18187 (1.8%)
ARICA,18693 (1.8%),14102 (2.0%),15728 (2.1%),17048 (2.0%),18207 (1.9%),18453 (1.9%)
...,...,...,...,...,...,...
PUTRE,3 (0.0%),0 (0.0%),2 (0.0%),2 (0.0%),5 (0.0%),4 (0.0%)
TORRES DEL PAINE,7 (0.0%),1 (0.0%),1 (0.0%),0 (0.0%),5 (0.0%),0 (0.0%)
GENERAL LAGOS,0 (0.0%),3 (0.0%),1 (0.0%),0 (0.0%),7 (0.0%),3 (0.0%)
CAMARONES,0 (0.0%),1 (0.0%),2 (0.0%),0 (0.0%),2 (0.0%),9 (0.0%)


### **7. Nacionalidad (NACIONALIDAD)**
- **Tipo de variable**: Demográfica.
- **Descripción**: País de origen o nacionalidad del paciente.
- **Cardinalidad**: Alta (214 categorías únicas reportadas, con errores de registro continental).
- **Veredicto**: **Transformar y Binarizar (`ES_CHILENO`)**. El análisis de frecuencias demuestra una varianza mínima extendida a lo largo de más de 200 categorías, donde la categoría "CHILE" acapara entre el 94% y 97% de los episodios hospitalarios. Para evitar la dispersión de datos y rescatar el valor sociológico de esta variable (la condición de migrante suele estar asociada a barreras de acceso, diagnósticos tardíos o diferencias en el consumo de recursos), se transformará en una variable dicotómica: `1` (Chileno) y `0` (Extranjero). Esta binarización condensa el impacto epidemiológico de la migración sin castigar a los selectores de características.

In [10]:
total, oncologico, control = distribucion_categorica_matriz(carpeta, archivos, mapa_columnas,"NACIONALIDAD")
mostrar_resultados(total, oncologico, control, "NACIONALIDAD")


[NACIONALIDAD] Extrayendo y procesando datos masivos (Una sola pasada)...
✓ Cardinalidad NACIONALIDAD: 214 categorías únicas
✓ Archivos CSV generados y guardados en las 3 carpetas correspondientes.


'1. Frecuencia para cohorte total (NACIONALIDAD):'

,2019,2020,2021,2022,2023,2024
CHILE,1086008 (94.3%),739048 (94.5%),769125 (94.2%),872642 (93.5%),975501 (93.8%),1019656 (93.9%)
VENEZUELA (REPUBLICA BOLIVARIANA DE),11625 (1.0%),10556 (1.4%),15055 (1.8%),22130 (2.4%),24510 (2.4%),24788 (2.3%)
PERU,7390 (0.6%),6916 (0.9%),8092 (1.0%),9297 (1.0%),9891 (1.0%),10428 (1.0%)
BOLIVIA (ESTADO PLURINACIONAL DE),5548 (0.5%),5689 (0.7%),7174 (0.9%),9497 (1.0%),10636 (1.0%),11680 (1.1%)
HAITI,11380 (1.0%),8804 (1.1%),8366 (1.0%),7589 (0.8%),5444 (0.5%),4383 (0.4%)
...,...,...,...,...,...,...
SEYCHELLES,0 (0.0%),1 (0.0%),0 (0.0%),0 (0.0%),0 (0.0%),0 (0.0%)
TANZANIA (REPUBLICA UNIDA DE),1 (0.0%),0 (0.0%),0 (0.0%),0 (0.0%),0 (0.0%),0 (0.0%)
SINGAPUR,0 (0.0%),0 (0.0%),1 (0.0%),0 (0.0%),0 (0.0%),0 (0.0%)
TURKMENISTAN,0 (0.0%),0 (0.0%),0 (0.0%),0 (0.0%),1 (0.0%),0 (0.0%)


'2. Frecuencia para cohorte oncológica (NACIONALIDAD):'

,2019,2020,2021,2022,2023,2024
CHILE,95325 (97.4%),63311 (97.5%),66099 (97.3%),75702 (97.0%),87576 (96.5%),95468 (96.3%)
VENEZUELA (REPUBLICA BOLIVARIANA DE),548 (0.6%),537 (0.8%),765 (1.1%),1027 (1.3%),1278 (1.4%),1651 (1.7%)
PERU,273 (0.3%),301 (0.5%),324 (0.5%),429 (0.5%),612 (0.7%),649 (0.7%)
COLOMBIA,168 (0.2%),187 (0.3%),198 (0.3%),273 (0.3%),440 (0.5%),426 (0.4%)
BOLIVIA (ESTADO PLURINACIONAL DE),217 (0.2%),155 (0.2%),219 (0.3%),232 (0.3%),311 (0.3%),427 (0.4%)
...,...,...,...,...,...,...
SANTA SEDE,0 (0.0%),0 (0.0%),1 (0.0%),0 (0.0%),0 (0.0%),0 (0.0%)
TERRITORIO BRITANICO DEL OCEANO INDICO,0 (0.0%),0 (0.0%),0 (0.0%),0 (0.0%),1 (0.0%),0 (0.0%)
TURQUIA,0 (0.0%),0 (0.0%),0 (0.0%),0 (0.0%),0 (0.0%),1 (0.0%)
VIETNAM,0 (0.0%),1 (0.0%),0 (0.0%),0 (0.0%),0 (0.0%),0 (0.0%)


'3. Frecuencia para cohorte control (NACIONALIDAD):'

,2019,2020,2021,2022,2023,2024
CHILE,990683 (94.0%),675737 (94.2%),703026 (93.9%),796940 (93.2%),887925 (93.6%),924188 (93.7%)
VENEZUELA (REPUBLICA BOLIVARIANA DE),11077 (1.1%),10019 (1.4%),14290 (1.9%),21103 (2.5%),23232 (2.4%),23137 (2.3%)
PERU,7117 (0.7%),6615 (0.9%),7768 (1.0%),8868 (1.0%),9279 (1.0%),9779 (1.0%)
BOLIVIA (ESTADO PLURINACIONAL DE),5331 (0.5%),5534 (0.8%),6955 (0.9%),9265 (1.1%),10325 (1.1%),11253 (1.1%)
HAITI,11267 (1.1%),8734 (1.2%),8301 (1.1%),7486 (0.9%),5360 (0.6%),4256 (0.4%)
...,...,...,...,...,...,...
SEYCHELLES,0 (0.0%),1 (0.0%),0 (0.0%),0 (0.0%),0 (0.0%),0 (0.0%)
TANZANIA (REPUBLICA UNIDA DE),1 (0.0%),0 (0.0%),0 (0.0%),0 (0.0%),0 (0.0%),0 (0.0%)
SINGAPUR,0 (0.0%),0 (0.0%),1 (0.0%),0 (0.0%),0 (0.0%),0 (0.0%)
TURKMENISTAN,0 (0.0%),0 (0.0%),0 (0.0%),0 (0.0%),1 (0.0%),0 (0.0%)


### **8. Previsión de Salud (PREVISION)**
- **Tipo de variable**: Demográfica / Socioeconómica.
- **Descripción**: Sistema de aseguramiento de salud al que está adscrito el paciente al momento del egreso.
- **Cardinalidad**: Moderada (14 categorías, con redundancias semánticas).
- **Veredicto**: **Retener y Agrupar**. Esta variable es el *proxy* (indicador indirecto) más robusto del nivel socioeconómico del paciente en el sistema chileno. Sin embargo, requiere una agrupación para reducir el ruido. Se consolidará en **7 macrocategorías funcionales**: los 4 tramos institucionales de riesgo socioeconómico (`FONASA A` [indigencia], `FONASA B`, `FONASA C`, `FONASA D` [mayor ingreso]), consolidando dentro de ellas sus respectivas modalidades de Libre Elección (FMLE); un grupo privado (`ISAPRE` y `PARTICULAR`, agrupados por representar pacientes fuera de la red pública habitual); un grupo de Fuerzas Armadas (`DIPRECA`, `CAPREDENA`, `SISA` -> `FFAA_Y_ORDEN`); y finalmente una categoría `DESCONOCIDO` (que absorberá nulos y "NO IDENTIFICADA").

In [11]:
total, oncologico, control = distribucion_categorica_matriz(carpeta, archivos, mapa_columnas,"PREVISION")
mostrar_resultados(total, oncologico, control, "PREVISION")


[PREVISION] Extrayendo y procesando datos masivos (Una sola pasada)...
✓ Cardinalidad PREVISION: 14 categorías únicas
✓ Archivos CSV generados y guardados en las 3 carpetas correspondientes.


'1. Frecuencia para cohorte total (PREVISION):'

,2019,2020,2021,2022,2023,2024
FONASA INSTITUCIONAL - (MAI) B,489103 (42.5%),347393 (44.4%),376291 (46.1%),439654 (47.1%),496115 (47.7%),529210 (48.7%)
FONASA INSTITUCIONAL - (MAI) A,308865 (26.8%),199295 (25.5%),195516 (23.9%),217676 (23.3%),229289 (22.1%),225893 (20.8%)
FONASA INSTITUCIONAL - (MAI) D,157463 (13.7%),110474 (14.1%),118309 (14.5%),129793 (13.9%),150708 (14.5%),159612 (14.7%)
FONASA INSTITUCIONAL - (MAI) C,127890 (11.1%),93205 (11.9%),98673 (12.1%),109294 (11.7%),130039 (12.5%),139120 (12.8%)
FONASA LIBRE ELECCION (FMLE_B),16092 (1.4%),6197 (0.8%),5566 (0.7%),9304 (1.0%),9741 (0.9%),9728 (0.9%)
ISAPRE,11364 (1.0%),7862 (1.0%),7500 (0.9%),7471 (0.8%),6482 (0.6%),5816 (0.5%)
PARTICULAR,15088 (1.3%),6500 (0.8%),6180 (0.8%),7553 (0.8%),5141 (0.5%),3533 (0.3%)
FONASA LIBRE ELECCION (FMLE_D),11219 (1.0%),4523 (0.6%),3651 (0.4%),5781 (0.6%),5496 (0.5%),5968 (0.5%)
FONASA LIBRE ELECCION (FMLE_C),6279 (0.5%),2128 (0.3%),1816 (0.2%),2782 (0.3%),2892 (0.3%),2963 (0.3%)
DIPRECA,1749 (0.2%),1823 (0.2%),1957 (0.2%),2228 (0.2%),2436 (0.2%),2595 (0.2%)


'2. Frecuencia para cohorte oncológica (PREVISION):'

,2019,2020,2021,2022,2023,2024
FONASA INSTITUCIONAL - (MAI) B,51366 (52.5%),35791 (55.1%),38243 (56.3%),45631 (58.4%),53563 (59.0%),59177 (59.7%)
FONASA INSTITUCIONAL - (MAI) A,18228 (18.6%),11893 (18.3%),11329 (16.7%),12460 (16.0%),13060 (14.4%),12598 (12.7%)
FONASA INSTITUCIONAL - (MAI) D,15527 (15.9%),9436 (14.5%),9812 (14.4%),10734 (13.7%),12734 (14.0%),14243 (14.4%)
FONASA INSTITUCIONAL - (MAI) C,10801 (11.0%),6916 (10.7%),7878 (11.6%),8405 (10.8%),10509 (11.6%),12259 (12.4%)
ISAPRE,330 (0.3%),205 (0.3%),213 (0.3%),222 (0.3%),254 (0.3%),202 (0.2%)
FONASA LIBRE ELECCION (FMLE_B),459 (0.5%),128 (0.2%),80 (0.1%),139 (0.2%),161 (0.2%),196 (0.2%)
PARTICULAR,248 (0.3%),176 (0.3%),126 (0.2%),168 (0.2%),93 (0.1%),115 (0.1%)
DIPRECA,104 (0.1%),148 (0.2%),111 (0.2%),129 (0.2%),153 (0.2%),161 (0.2%)
FONASA LIBRE ELECCION (FMLE_D),218 (0.2%),66 (0.1%),35 (0.1%),58 (0.1%),76 (0.1%),63 (0.1%)
NO IDENTIFICADA,297 (0.3%),34 (0.1%),1 (0.0%),0 (0.0%),0 (0.0%),0 (0.0%)


'3. Frecuencia para cohorte control (PREVISION):'

,2019,2020,2021,2022,2023,2024
FONASA INSTITUCIONAL - (MAI) B,437737 (41.5%),311602 (43.5%),338048 (45.1%),394023 (46.1%),442552 (46.6%),470033 (47.6%)
FONASA INSTITUCIONAL - (MAI) A,290637 (27.6%),187402 (26.1%),184187 (24.6%),205216 (24.0%),216229 (22.8%),213295 (21.6%)
FONASA INSTITUCIONAL - (MAI) D,141936 (13.5%),101038 (14.1%),108497 (14.5%),119059 (13.9%),137974 (14.5%),145369 (14.7%)
FONASA INSTITUCIONAL - (MAI) C,117089 (11.1%),86289 (12.0%),90795 (12.1%),100889 (11.8%),119530 (12.6%),126861 (12.9%)
FONASA LIBRE ELECCION (FMLE_B),15633 (1.5%),6069 (0.8%),5486 (0.7%),9165 (1.1%),9580 (1.0%),9532 (1.0%)
ISAPRE,11034 (1.0%),7657 (1.1%),7287 (1.0%),7249 (0.8%),6228 (0.7%),5614 (0.6%)
PARTICULAR,14840 (1.4%),6324 (0.9%),6054 (0.8%),7385 (0.9%),5048 (0.5%),3418 (0.3%)
FONASA LIBRE ELECCION (FMLE_D),11001 (1.0%),4457 (0.6%),3616 (0.5%),5723 (0.7%),5420 (0.6%),5905 (0.6%)
FONASA LIBRE ELECCION (FMLE_C),6135 (0.6%),2093 (0.3%),1806 (0.2%),2744 (0.3%),2860 (0.3%),2921 (0.3%)
DIPRECA,1645 (0.2%),1675 (0.2%),1846 (0.2%),2099 (0.2%),2283 (0.2%),2434 (0.2%)


### **9. Servicio de Salud (SERVICIO_SALUD)**
- **Tipo de variable**: Hospitalaria / Administrativa.
- **Descripción**: Red macro-zonal del sistema de salud público a la que pertenece o fue derivado el paciente.
- **Cardinalidad**: Manejable (31 categorías válidas).
- **Veredicto**: **Retener y Limpiar**. A diferencia de la comuna, 31 categorías representan una cardinalidad perfectamente asimilable por los selectores de características y los algoritmos basados en árboles. Esta variable es de suma importancia estratégica, ya que captura las asimetrías institucionales del país: diferencias de infraestructura, disponibilidad de centros oncológicos de referencia, presupuestos zonales y tiempos en listas de espera. Solo se aplicará una limpieza básica unificando las etiquetas anómalas (`NO APLICA`, `IGNORADO`) dentro de la categoría `DESCONOCIDO`.

In [12]:
total, oncologico, control = distribucion_categorica_matriz(carpeta, archivos, mapa_columnas,"SERVICIO_SALUD")
mostrar_resultados(total, oncologico, control, "SERVICIO_SALUD")


[SERVICIO_SALUD] Extrayendo y procesando datos masivos (Una sola pasada)...
✓ Cardinalidad SERVICIO_SALUD: 31 categorías únicas
✓ Archivos CSV generados y guardados en las 3 carpetas correspondientes.


'1. Frecuencia para cohorte total (SERVICIO_SALUD):'

,2019,2020,2021,2022,2023,2024
METROPOLITANO SURORIENTE,100291 (8.7%),70122 (9.0%),79850 (9.8%),89304 (9.6%),93199 (9.0%),90733 (8.4%)
DEL MAULE,76222 (6.6%),54441 (7.0%),54716 (6.7%),65501 (7.0%),70645 (6.8%),82844 (7.6%)
METROPOLITANO OCCIDENTE,70647 (6.1%),48504 (6.2%),51165 (6.3%),61458 (6.6%),70997 (6.8%),73247 (6.7%)
METROPOLITANO SUR,66311 (5.8%),51253 (6.6%),51210 (6.3%),54863 (5.9%),55839 (5.4%),59892 (5.5%)
METROPOLITANO CENTRAL,65480 (5.7%),45189 (5.8%),43943 (5.4%),50325 (5.4%),54484 (5.2%),54335 (5.0%)
ARAUCANIA SUR,57754 (5.0%),40467 (5.2%),38203 (4.7%),41344 (4.4%),46871 (4.5%),58777 (5.4%)
COQUIMBO,51921 (4.5%),34611 (4.4%),37929 (4.6%),43628 (4.7%),52101 (5.0%),52366 (4.8%)
VINA DEL MAR QUILLOTA,59593 (5.2%),33626 (4.3%),32698 (4.0%),37432 (4.0%),50370 (4.8%),51940 (4.8%)
LIBERTADOR B. O HIGGINS,51731 (4.5%),35614 (4.6%),37340 (4.6%),43614 (4.7%),44749 (4.3%),45492 (4.2%)
METROPOLITANO ORIENTE,48966 (4.3%),32609 (4.2%),33731 (4.1%),37664 (4.0%),40669 (3.9%),40694 (3.7%)


'2. Frecuencia para cohorte oncológica (SERVICIO_SALUD):'

,2019,2020,2021,2022,2023,2024
METROPOLITANO SURORIENTE,12838 (13.1%),6619 (10.2%),7923 (11.7%),8736 (11.2%),9363 (10.3%),9573 (9.7%)
METROPOLITANO OCCIDENTE,5752 (5.9%),4041 (6.2%),5104 (7.5%),5911 (7.6%),6750 (7.4%),6961 (7.0%)
DEL MAULE,5802 (5.9%),4334 (6.7%),4448 (6.5%),5060 (6.5%),5618 (6.2%),6545 (6.6%)
METROPOLITANO CENTRAL,5744 (5.9%),4154 (6.4%),3407 (5.0%),4488 (5.7%),5472 (6.0%),5703 (5.8%)
VINA DEL MAR QUILLOTA,7342 (7.5%),2940 (4.5%),2505 (3.7%),3286 (4.2%),4720 (5.2%),4910 (5.0%)
CONCEPCION,4103 (4.2%),3143 (4.8%),3497 (5.1%),4008 (5.1%),4864 (5.4%),5306 (5.4%)
METROPOLITANO ORIENTE,4685 (4.8%),2957 (4.6%),3310 (4.9%),3852 (4.9%),4569 (5.0%),4948 (5.0%)
ARAUCANIA SUR,4448 (4.5%),3465 (5.3%),3150 (4.6%),3275 (4.2%),4055 (4.5%),5190 (5.2%)
METROPOLITANO SUR,4207 (4.3%),3431 (5.3%),3409 (5.0%),3674 (4.7%),4037 (4.5%),4537 (4.6%)
VALPARAISO SAN ANTONIO,6662 (6.8%),2476 (3.8%),2377 (3.5%),2854 (3.7%),3554 (3.9%),3872 (3.9%)


'3. Frecuencia para cohorte control (SERVICIO_SALUD):'

,2019,2020,2021,2022,2023,2024
METROPOLITANO SURORIENTE,87453 (8.3%),63503 (8.9%),71927 (9.6%),80568 (9.4%),83836 (8.8%),81160 (8.2%)
DEL MAULE,70420 (6.7%),50107 (7.0%),50268 (6.7%),60441 (7.1%),65027 (6.9%),76299 (7.7%)
METROPOLITANO OCCIDENTE,64895 (6.2%),44463 (6.2%),46061 (6.1%),55547 (6.5%),64247 (6.8%),66286 (6.7%)
METROPOLITANO SUR,62104 (5.9%),47822 (6.7%),47801 (6.4%),51189 (6.0%),51802 (5.5%),55355 (5.6%)
METROPOLITANO CENTRAL,59736 (5.7%),41035 (5.7%),40536 (5.4%),45837 (5.4%),49012 (5.2%),48632 (4.9%)
ARAUCANIA SUR,53306 (5.1%),37002 (5.2%),35053 (4.7%),38069 (4.5%),42816 (4.5%),53587 (5.4%)
COQUIMBO,48433 (4.6%),32033 (4.5%),34935 (4.7%),40198 (4.7%),47958 (5.1%),47925 (4.9%)
LIBERTADOR B. O HIGGINS,48664 (4.6%),33008 (4.6%),34619 (4.6%),40404 (4.7%),41381 (4.4%),41965 (4.3%)
VINA DEL MAR QUILLOTA,52251 (5.0%),30686 (4.3%),30193 (4.0%),34146 (4.0%),45650 (4.8%),47030 (4.8%)
METROPOLITANO ORIENTE,44281 (4.2%),29652 (4.1%),30421 (4.1%),33812 (4.0%),36100 (3.8%),35746 (3.6%)


### **10. Tipo de procedencia (TIPO_PROCEDENCIA)**
- **Tipo de variable**: Hospitalaria / Administrativa.
- **Descripción**: Indica el origen físico o institucional del paciente inmediatamente antes de ingresar al hospital.
- **Cardinalidad**: Moderada (19 categorías únicas).
- **Veredicto**: **Retener y Agrupar**. El análisis revela un hallazgo estructural crítico para la caracterización: mientras que en el grupo control la mayoría ingresa desde el `SERVICIO DE EMERGENCIA` (~45-54%), el perfil oncológico es radicalmente distinto, proviniendo mayoritariamente de `CENTRO DE ESPECIALIDADES (CDT, CRS)` (~55-64%). Para optimizar su uso en los selectores de características, la variable se agrupará en 5 macro-categorías funcionales que reflejen la vía de acceso: *Atención Primaria / Ambulatoria*, *Urgencia*, *Derivación de Red Hospitalaria*, *Sector Privado* y *Otros*.

In [13]:
total, oncologico, control = distribucion_categorica_matriz(carpeta, archivos, mapa_columnas,"TIPO_PROCEDENCIA")
mostrar_resultados(total, oncologico, control, "TIPO_PROCEDENCIA")


[TIPO_PROCEDENCIA] Extrayendo y procesando datos masivos (Una sola pasada)...
✓ Cardinalidad TIPO_PROCEDENCIA: 19 categorías únicas
✓ Archivos CSV generados y guardados en las 3 carpetas correspondientes.


'1. Frecuencia para cohorte total (TIPO_PROCEDENCIA):'

,2019,2020,2021,2022,2023,2024
SERVICIO EMERGENCIA (DOMICILIO),532216 (46.2%),410739 (52.5%),409190 (50.1%),436754 (46.8%),464982 (44.7%),475146 (43.8%)
"CENTRO ESPECIALIDADES (CDT, CRS, CONSULTORIO ADOS. ESP)",408511 (35.5%),206419 (26.4%),227889 (27.9%),282818 (30.3%),346334 (33.3%),369242 (34.0%)
OTROS HOSPITALES DE LA RED,71130 (6.2%),62884 (8.0%),63987 (7.8%),63660 (6.8%),71394 (6.9%),78041 (7.2%)
"APS URGENCIA (SAPU, SUR, SUC)",37707 (3.3%),36944 (4.7%),45545 (5.6%),44039 (4.7%),50490 (4.9%),56485 (5.2%)
CONSULTA PRIVADA,38950 (3.4%),16065 (2.1%),14804 (1.8%),22300 (2.4%),23209 (2.2%),23809 (2.2%)
APS CONSULTORIO (CESFAM),19926 (1.7%),15284 (2.0%),13841 (1.7%),12893 (1.4%),14612 (1.4%),15408 (1.4%)
"OTRAS INSTITUCIONES SALUD (CLINICAS PRIVADAS, DE REHABILITAC",13108 (1.1%),11589 (1.5%),13066 (1.6%),11555 (1.2%),11697 (1.1%),12267 (1.1%)
OTROS HOSPITALES RED NACIONAL,13647 (1.2%),10928 (1.4%),11475 (1.4%),10552 (1.1%),11783 (1.1%),11722 (1.1%)
PLAN DE RESOLUCION LE,0 (0.0%),0 (0.0%),0 (0.0%),31745 (3.4%),16937 (1.6%),57 (0.0%)
ESTRATEGIA CRR,0 (0.0%),0 (0.0%),0 (0.0%),3235 (0.3%),10423 (1.0%),22826 (2.1%)


'2. Frecuencia para cohorte oncológica (TIPO_PROCEDENCIA):'

,2019,2020,2021,2022,2023,2024
"CENTRO ESPECIALIDADES (CDT, CRS, CONSULTORIO ADOS. ESP)",62367 (63.8%),34814 (53.6%),37940 (55.9%),43489 (55.7%),50023 (55.1%),54557 (55.0%)
SERVICIO EMERGENCIA (DOMICILIO),25581 (26.1%),21339 (32.9%),21105 (31.1%),23550 (30.2%),27719 (30.6%),29911 (30.2%)
OTROS HOSPITALES DE LA RED,3600 (3.7%),3407 (5.2%),3245 (4.8%),3435 (4.4%),4416 (4.9%),5175 (5.2%)
"APS URGENCIA (SAPU, SUR, SUC)",1515 (1.5%),1583 (2.4%),1861 (2.7%),2130 (2.7%),2414 (2.7%),3038 (3.1%)
OTROS HOSPITALES RED NACIONAL,1137 (1.2%),905 (1.4%),825 (1.2%),875 (1.1%),911 (1.0%),1034 (1.0%)
"OTRAS INSTITUCIONES SALUD (CLINICAS PRIVADAS, DE REHABILITAC",836 (0.9%),827 (1.3%),823 (1.2%),811 (1.0%),991 (1.1%),1030 (1.0%)
CONSULTA PRIVADA,1246 (1.3%),581 (0.9%),555 (0.8%),702 (0.9%),771 (0.8%),818 (0.8%)
APS CONSULTORIO (CESFAM),783 (0.8%),682 (1.1%),578 (0.9%),566 (0.7%),684 (0.8%),777 (0.8%)
"OTRAS INSTITUCIONES (CARCEL, HOGARES DE ANCIANOS, SENAME, EC",105 (0.1%),274 (0.4%),234 (0.3%),181 (0.2%),606 (0.7%),849 (0.9%)
PLAN DE RESOLUCION LE,0 (0.0%),0 (0.0%),0 (0.0%),1468 (1.9%),777 (0.9%),2 (0.0%)


'3. Frecuencia para cohorte control (TIPO_PROCEDENCIA):'

,2019,2020,2021,2022,2023,2024
SERVICIO EMERGENCIA (DOMICILIO),506635 (48.1%),389400 (54.3%),388085 (51.8%),413204 (48.3%),437263 (46.1%),445235 (45.1%)
"CENTRO ESPECIALIDADES (CDT, CRS, CONSULTORIO ADOS. ESP)",346144 (32.9%),171605 (23.9%),189949 (25.4%),239329 (28.0%),296311 (31.2%),314685 (31.9%)
OTROS HOSPITALES DE LA RED,67530 (6.4%),59477 (8.3%),60742 (8.1%),60225 (7.0%),66978 (7.1%),72866 (7.4%)
"APS URGENCIA (SAPU, SUR, SUC)",36192 (3.4%),35361 (4.9%),43684 (5.8%),41909 (4.9%),48076 (5.1%),53447 (5.4%)
CONSULTA PRIVADA,37704 (3.6%),15484 (2.2%),14249 (1.9%),21598 (2.5%),22438 (2.4%),22991 (2.3%)
APS CONSULTORIO (CESFAM),19143 (1.8%),14602 (2.0%),13263 (1.8%),12327 (1.4%),13928 (1.5%),14631 (1.5%)
"OTRAS INSTITUCIONES SALUD (CLINICAS PRIVADAS, DE REHABILITAC",12272 (1.2%),10762 (1.5%),12243 (1.6%),10744 (1.3%),10706 (1.1%),11237 (1.1%)
OTROS HOSPITALES RED NACIONAL,12510 (1.2%),10023 (1.4%),10650 (1.4%),9677 (1.1%),10872 (1.1%),10688 (1.1%)
PLAN DE RESOLUCION LE,0 (0.0%),0 (0.0%),0 (0.0%),30277 (3.5%),16160 (1.7%),55 (0.0%)
ESTRATEGIA CRR,0 (0.0%),0 (0.0%),0 (0.0%),3136 (0.4%),9924 (1.0%),21946 (2.2%)


### **11. Tipo de ingreso (TIPO_INGRESO)**
- **Tipo de variable**: Hospitalaria / Administrativa.
- **Descripción**: Clasificación administrativa del ingreso del paciente al episodio asistencial.
- **Cardinalidad**: Baja (5 categorías únicas).
- **Veredicto**: **Retener**. Presenta una cardinalidad casi perfecta y expone otra diferencia epidemiológica fundamental: el ~56-65% de los ingresos oncológicos son `PROGRAMADOS`, en contraste directo con el grupo control, que está dominado por la `URGENCIA` (~48-57%). Se mantendrán las tres categorías principales (`URGENCIA`, `PROGRAMADA`, `OBSTETRICA`), y los valores anómalos o marginales (`NO PROGRAMADA`, `NO IDENTIFICADA`, `DESCONOCIDO`) se imputarán a la moda o se unificarán.

In [14]:
total, oncologico, control = distribucion_categorica_matriz(carpeta, archivos, mapa_columnas,"TIPO_INGRESO")
mostrar_resultados(total, oncologico, control, "TIPO_INGRESO")


[TIPO_INGRESO] Extrayendo y procesando datos masivos (Una sola pasada)...
✓ Cardinalidad TIPO_INGRESO: 5 categorías únicas
✓ Archivos CSV generados y guardados en las 3 carpetas correspondientes.


'1. Frecuencia para cohorte total (TIPO_INGRESO):'

,2019,2020,2021,2022,2023,2024
URGENCIA,548049 (47.6%),432217 (55.3%),457702 (56.0%),468530 (50.2%),518634 (49.9%),552102 (50.8%)
PROGRAMADA,421644 (36.6%),199311 (25.5%),220715 (27.0%),307367 (32.9%),366371 (35.2%),388739 (35.8%)
OBSTETRICA,181372 (15.8%),150360 (19.2%),138491 (17.0%),156907 (16.8%),154527 (14.9%),144929 (13.3%)
NO IDENTIFICADA,299 (0.0%),18 (0.0%),1 (0.0%),0 (0.0%),0 (0.0%),0 (0.0%)
DESCONOCIDO,14 (0.0%),0 (0.0%),0 (0.0%),33 (0.0%),55 (0.0%),43 (0.0%)
NO PROGRAMADA,20 (0.0%),5 (0.0%),0 (0.0%),0 (0.0%),0 (0.0%),0 (0.0%)


'2. Frecuencia para cohorte oncológica (TIPO_INGRESO):'

,2019,2020,2021,2022,2023,2024
PROGRAMADA,63995 (65.4%),36431 (56.1%),38187 (56.2%),45085 (57.8%),52206 (57.5%),55746 (56.2%)
URGENCIA,33623 (34.4%),28275 (43.5%),29540 (43.5%),32677 (41.9%),38203 (42.1%),43027 (43.4%)
OBSTETRICA,197 (0.2%),220 (0.3%),199 (0.3%),302 (0.4%),305 (0.3%),371 (0.4%)
DESCONOCIDO,0 (0.0%),0 (0.0%),0 (0.0%),5 (0.0%),1 (0.0%),9 (0.0%)
NO IDENTIFICADA,14 (0.0%),0 (0.0%),0 (0.0%),0 (0.0%),0 (0.0%),0 (0.0%)


'3. Frecuencia para cohorte control (TIPO_INGRESO):'

,2019,2020,2021,2022,2023,2024
URGENCIA,514426 (48.8%),403942 (56.3%),428162 (57.2%),435853 (51.0%),480431 (50.6%),509075 (51.6%)
PROGRAMADA,357649 (33.9%),162880 (22.7%),182528 (24.4%),262282 (30.7%),314165 (33.1%),332993 (33.7%)
OBSTETRICA,181175 (17.2%),150140 (20.9%),138292 (18.5%),156605 (18.3%),154222 (16.3%),144558 (14.7%)
NO IDENTIFICADA,285 (0.0%),18 (0.0%),1 (0.0%),0 (0.0%),0 (0.0%),0 (0.0%)
DESCONOCIDO,14 (0.0%),0 (0.0%),0 (0.0%),28 (0.0%),54 (0.0%),34 (0.0%)
NO PROGRAMADA,20 (0.0%),5 (0.0%),0 (0.0%),0 (0.0%),0 (0.0%),0 (0.0%)


### **12. Especialidad Médica (ESPECIALIDAD_MEDICA)**
- **Tipo de variable**: Clínica / Hospitalaria.
- **Descripción**: Especialidad médica principal encargada del tratamiento del paciente durante su episodio.
- **Cardinalidad**: Alta (103 categorías únicas, afectadas por cambios de nomenclatura).
- **Veredicto**: **Retener, Limpiar y Agrupar**. Dado que el estudio es de caracterización retrospectiva y no predictivo-prospectivo, no existe riesgo de fuga de información (*Data Leakage*). La alta cardinalidad (103) se debe principalmente a errores tipográficos y saltos en la codificación del MINSAL a partir de 2020 (ej. `CIRUGIA TORAX` vs. `CIRUGIA DE TORAX`). Se desarrollará un diccionario de homologación para fusionar las especialidades equivalentes y condensarlas en ~12 macro-especialidades (ej. *Cirugía*, *Medicina Interna*, *Oncología*, *Ginecología y Obstetricia*), estabilizando la variable para su análisis mediante *Bootstrapping*.

In [15]:
total, oncologico, control = distribucion_categorica_matriz(carpeta, archivos, mapa_columnas,"ESPECIALIDAD_MEDICA")
mostrar_resultados(total, oncologico, control, "ESPECIALIDAD_MEDICA")


[ESPECIALIDAD_MEDICA] Extrayendo y procesando datos masivos (Una sola pasada)...
✓ Cardinalidad ESPECIALIDAD_MEDICA: 103 categorías únicas
✓ Archivos CSV generados y guardados en las 3 carpetas correspondientes.


'1. Frecuencia para cohorte total (ESPECIALIDAD_MEDICA):'

,2019,2020,2021,2022,2023,2024
OBSTETRICIA Y GINECOLOGIA,234462 (20.4%),179958 (23.0%),173148 (21.2%),202672 (21.7%),208926 (20.1%),205359 (18.9%)
CIRUGIA GENERAL,185431 (16.1%),124137 (15.9%),120050 (14.7%),137340 (14.7%),149835 (14.4%),162334 (15.0%)
MEDICINA INTERNA,142928 (12.4%),135740 (17.4%),147449 (18.0%),131421 (14.1%),140730 (13.5%),155574 (14.3%)
TRAUMATOLOGIA Y ORTOPEDIA,83285 (7.2%),57434 (7.3%),62207 (7.6%),75259 (8.1%),90911 (8.7%),93848 (8.6%)
PEDIATRIA,78098 (6.8%),35330 (4.5%),40002 (4.9%),59484 (6.4%),71680 (6.9%),68541 (6.3%)
...,...,...,...,...,...,...
IMAGENOLOGIA,0 (0.0%),0 (0.0%),1 (0.0%),0 (0.0%),0 (0.0%),1 (0.0%)
SALUD PUBLICA (ODONTOLOGICA),0 (0.0%),0 (0.0%),0 (0.0%),0 (0.0%),0 (0.0%),2 (0.0%)
IMAGENOLOGIA ORAL Y MAXILOFACIAL,0 (0.0%),1 (0.0%),0 (0.0%),0 (0.0%),0 (0.0%),0 (0.0%)
MEDICINA MATERNO FETAL,0 (0.0%),1 (0.0%),0 (0.0%),0 (0.0%),0 (0.0%),0 (0.0%)


'2. Frecuencia para cohorte oncológica (ESPECIALIDAD_MEDICA):'

,2019,2020,2021,2022,2023,2024
CIRUGIA GENERAL,22150 (22.6%),16872 (26.0%),17550 (25.8%),19433 (24.9%),21464 (23.7%),22244 (22.4%)
MEDICINA INTERNA,15681 (16.0%),13180 (20.3%),12845 (18.9%),14631 (18.7%),16994 (18.7%),19370 (19.5%)
ONCOLOGIA MEDICA,21996 (22.5%),7599 (11.7%),6281 (9.2%),7148 (9.2%),8183 (9.0%),9200 (9.3%)
OBSTETRICIA Y GINECOLOGIA,6971 (7.1%),5544 (8.5%),6035 (8.9%),7131 (9.1%),8033 (8.9%),8825 (8.9%)
UROLOGIA,6021 (6.2%),4632 (7.1%),5298 (7.8%),6267 (8.0%),7513 (8.3%),8063 (8.1%)
...,...,...,...,...,...,...
INFECTOLOGIA PEDIATRICA,0 (0.0%),2 (0.0%),0 (0.0%),0 (0.0%),0 (0.0%),0 (0.0%)
GINECOLOGIA PEDIATRICA Y DE LA ADOLESCENCIA,0 (0.0%),0 (0.0%),0 (0.0%),1 (0.0%),0 (0.0%),1 (0.0%)
MATRONAS(ES),0 (0.0%),0 (0.0%),0 (0.0%),1 (0.0%),0 (0.0%),1 (0.0%)
ORTODONCIA Y ORTOPEDIA DENTO MAXILOFACIAL,0 (0.0%),1 (0.0%),1 (0.0%),0 (0.0%),0 (0.0%),0 (0.0%)


'3. Frecuencia para cohorte control (ESPECIALIDAD_MEDICA):'

,2019,2020,2021,2022,2023,2024
OBSTETRICIA Y GINECOLOGIA,227491 (21.6%),174414 (24.3%),167113 (22.3%),195541 (22.9%),200893 (21.2%),196534 (19.9%)
MEDICINA INTERNA,127247 (12.1%),122560 (17.1%),134604 (18.0%),116790 (13.7%),123736 (13.0%),136204 (13.8%)
CIRUGIA GENERAL,163281 (15.5%),107265 (15.0%),102500 (13.7%),117907 (13.8%),128371 (13.5%),140090 (14.2%)
TRAUMATOLOGIA Y ORTOPEDIA,82140 (7.8%),56504 (7.9%),61121 (8.2%),73978 (8.7%),89156 (9.4%),91876 (9.3%)
PEDIATRIA,76950 (7.3%),34323 (4.8%),38851 (5.2%),58326 (6.8%),70563 (7.4%),67351 (6.8%)
...,...,...,...,...,...,...
IMAGENOLOGIA,0 (0.0%),0 (0.0%),1 (0.0%),0 (0.0%),0 (0.0%),1 (0.0%)
SALUD PUBLICA (ODONTOLOGICA),0 (0.0%),0 (0.0%),0 (0.0%),0 (0.0%),0 (0.0%),2 (0.0%)
IMAGENOLOGIA ORAL Y MAXILOFACIAL,0 (0.0%),1 (0.0%),0 (0.0%),0 (0.0%),0 (0.0%),0 (0.0%)
MEDICINA MATERNO FETAL,0 (0.0%),1 (0.0%),0 (0.0%),0 (0.0%),0 (0.0%),0 (0.0%)


### **13. Tipo de Actividad (TIPO_ACTIVIDAD)**
- **Tipo de variable**: Hospitalaria.
- **Descripción**: Modalidad de atención asociada al episodio clínico.
- **Cardinalidad**: Baja (5 categorías, con inconsistencias temporales).
- **Veredicto**: **Retener y Homologar**. Es un fuerte determinante del consumo de recursos. Sin embargo, el análisis detectó un quiebre en el registro: las categorías `HOSPITALIZACION DIURNA` y `HOSPITALIZACION EN URGENCIA` dejan de utilizarse a partir de 2020. Para garantizar la coherencia longitudinal del estudio a lo largo de los 6 años, estas categorías descontinuadas serán fusionadas dentro de la categoría principal `HOSPITALIZACION`. La variable quedará binarizada en la práctica entre hospitalización clásica y `CIRUGIA MAYOR AMBULATORIA (CMA)`.

In [16]:
total, oncologico, control = distribucion_categorica_matriz(carpeta, archivos, mapa_columnas,"TIPO_ACTIVIDAD")
mostrar_resultados(total, oncologico, control, "TIPO_ACTIVIDAD")


[TIPO_ACTIVIDAD] Extrayendo y procesando datos masivos (Una sola pasada)...
✓ Cardinalidad TIPO_ACTIVIDAD: 5 categorías únicas
✓ Archivos CSV generados y guardados en las 3 carpetas correspondientes.


'1. Frecuencia para cohorte total (TIPO_ACTIVIDAD):'

,2019,2020,2021,2022,2023,2024
HOSPITALIZACION,886571 (77.0%),702275 (89.8%),713978 (87.4%),778072 (83.4%),846043 (81.4%),873772 (80.5%)
CIRUGIA MAYOR AMBULATORIA (CMA),146928 (12.8%),79636 (10.2%),102931 (12.6%),154765 (16.6%),193544 (18.6%),212041 (19.5%)
HOSPITALIZACION EN URGENCIA,62551 (5.4%),0 (0.0%),0 (0.0%),0 (0.0%),0 (0.0%),0 (0.0%)
HOSPITALIZACION DIURNA,55327 (4.8%),0 (0.0%),0 (0.0%),0 (0.0%),0 (0.0%),0 (0.0%)
DESCONOCIDO,18 (0.0%),0 (0.0%),0 (0.0%),0 (0.0%),0 (0.0%),0 (0.0%)
NO IDENTIFICADO,3 (0.0%),0 (0.0%),0 (0.0%),0 (0.0%),0 (0.0%),0 (0.0%)


'2. Frecuencia para cohorte oncológica (TIPO_ACTIVIDAD):'

,2019,2020,2021,2022,2023,2024
HOSPITALIZACION,73808 (75.4%),61278 (94.4%),62237 (91.6%),70160 (89.9%),80500 (88.7%),88072 (88.8%)
CIRUGIA MAYOR AMBULATORIA (CMA),4502 (4.6%),3648 (5.6%),5689 (8.4%),7909 (10.1%),10215 (11.3%),11081 (11.2%)
HOSPITALIZACION DIURNA,16169 (16.5%),0 (0.0%),0 (0.0%),0 (0.0%),0 (0.0%),0 (0.0%)
HOSPITALIZACION EN URGENCIA,3350 (3.4%),0 (0.0%),0 (0.0%),0 (0.0%),0 (0.0%),0 (0.0%)


'3. Frecuencia para cohorte control (TIPO_ACTIVIDAD):'

,2019,2020,2021,2022,2023,2024
HOSPITALIZACION,812763 (77.1%),640997 (89.4%),651741 (87.0%),707912 (82.8%),765543 (80.7%),785700 (79.6%)
CIRUGIA MAYOR AMBULATORIA (CMA),142426 (13.5%),75988 (10.6%),97242 (13.0%),146856 (17.2%),183329 (19.3%),200960 (20.4%)
HOSPITALIZACION EN URGENCIA,59201 (5.6%),0 (0.0%),0 (0.0%),0 (0.0%),0 (0.0%),0 (0.0%)
HOSPITALIZACION DIURNA,39158 (3.7%),0 (0.0%),0 (0.0%),0 (0.0%),0 (0.0%),0 (0.0%)
DESCONOCIDO,18 (0.0%),0 (0.0%),0 (0.0%),0 (0.0%),0 (0.0%),0 (0.0%)
NO IDENTIFICADO,3 (0.0%),0 (0.0%),0 (0.0%),0 (0.0%),0 (0.0%),0 (0.0%)


### **14. Servicio de Ingreso (SERVICIOINGRESO)**
- **Tipo de variable**: Hospitalaria / Operativa.
- **Descripción**: Servicio clínico físico donde el paciente es admitido al iniciar su episodio.
- **Cardinalidad**: Alta (92 categorías únicas con alta redundancia).
- **Veredicto**: **Retener y Agrupar por Complejidad**. Esta variable detalla la carga de infraestructura física del hospital. Al igual que con las especialidades, presenta una alta fragmentación por digitación redundante (ej. `UTI ADULTOS` vs. `UNIDAD DE TRATAMIENTO INTERMEDIO...`). Se aplicará un procesamiento de lenguaje natural (NLP) o mapeo manual estructurado para agrupar estas 92 áreas en niveles de complejidad resolutiva: *Cuidados Críticos (UCI/UTI)*, *Pabellón Quirúrgico / Recuperación*, *Cuidados Medios/Básicos*, y *Maternidad/Pediatría*. Esto reducirá la dimensionalidad previniendo la dispersión de los algoritmos de selección de características.

In [17]:
total, oncologico, control = distribucion_categorica_matriz(carpeta, archivos, mapa_columnas,"SERVICIOINGRESO")
mostrar_resultados(total, oncologico, control, "SERVICIOINGRESO")


[SERVICIOINGRESO] Extrayendo y procesando datos masivos (Una sola pasada)...
✓ Cardinalidad SERVICIOINGRESO: 92 categorías únicas
✓ Archivos CSV generados y guardados en las 3 carpetas correspondientes.


'1. Frecuencia para cohorte total (SERVICIOINGRESO):'

,2019,2020,2021,2022,2023,2024
CIRUGIA,132161 (11.5%),91125 (11.7%),84442 (10.3%),95357 (10.2%),99893 (9.6%),101990 (9.4%)
UNIDAD DE RECUPERACION DE PABELLONES (CENTRAL Y CMA),101201 (8.8%),49633 (6.3%),63071 (7.7%),97241 (10.4%),125125 (12.0%),149515 (13.8%)
MEDICINA,101979 (8.9%),90153 (11.5%),86873 (10.6%),79883 (8.6%),81786 (7.9%),93439 (8.6%)
OBSTETRICIA,108329 (9.4%),88038 (11.3%),79730 (9.8%),86407 (9.3%),82838 (8.0%),74134 (6.8%)
PEDIATRIA,60846 (5.3%),30102 (3.8%),33581 (4.1%),42819 (4.6%),49442 (4.8%),46731 (4.3%)
...,...,...,...,...,...,...
PSIQUIATRIA FORENSE ALTA COMPLEJIDAD,0 (0.0%),0 (0.0%),0 (0.0%),0 (0.0%),1 (0.0%),9 (0.0%)
PSIQUIATRIA CRONICO,2 (0.0%),2 (0.0%),0 (0.0%),0 (0.0%),0 (0.0%),0 (0.0%)
CARDIOCIRUGIA (INFANTIL),1 (0.0%),1 (0.0%),1 (0.0%),0 (0.0%),0 (0.0%),0 (0.0%)
NULOS,0 (0.0%),2 (0.0%),0 (0.0%),0 (0.0%),0 (0.0%),0 (0.0%)


'2. Frecuencia para cohorte oncológica (SERVICIOINGRESO):'

,2019,2020,2021,2022,2023,2024
CIRUGIA,18271 (18.7%),13833 (21.3%),13699 (20.2%),14972 (19.2%),15891 (17.5%),15686 (15.8%)
MEDICINA,13323 (13.6%),11003 (16.9%),9106 (13.4%),9791 (12.5%),11496 (12.7%),13995 (14.1%)
ONCOLOGIA,16858 (17.2%),7171 (11.0%),5960 (8.8%),6052 (7.8%),6928 (7.6%),7369 (7.4%)
GINECOLOGIA,6685 (6.8%),5334 (8.2%),6095 (9.0%),6943 (8.9%),7704 (8.5%),7815 (7.9%)
UNIDAD DE RECUPERACION DE PABELLONES (CENTRAL Y CMA),3789 (3.9%),2396 (3.7%),3588 (5.3%),4761 (6.1%),5944 (6.6%),6996 (7.1%)
...,...,...,...,...,...,...
UNIDAD DE TRATAMIENTO INTERMEDIO QUEMADOS,1 (0.0%),0 (0.0%),1 (0.0%),1 (0.0%),0 (0.0%),0 (0.0%)
NEONATOLOGIA CUNAS,1 (0.0%),1 (0.0%),0 (0.0%),1 (0.0%),0 (0.0%),0 (0.0%)
AREANEONATOLOGIA CUIDADOS BASICOS,0 (0.0%),0 (0.0%),1 (0.0%),0 (0.0%),0 (0.0%),0 (0.0%)
NULOS,0 (0.0%),1 (0.0%),0 (0.0%),0 (0.0%),0 (0.0%),0 (0.0%)


'3. Frecuencia para cohorte control (SERVICIOINGRESO):'

,2019,2020,2021,2022,2023,2024
UNIDAD DE RECUPERACION DE PABELLONES (CENTRAL Y CMA),97412 (9.2%),47237 (6.6%),59483 (7.9%),92480 (10.8%),119181 (12.6%),142519 (14.4%)
OBSTETRICIA,108031 (10.3%),87815 (12.2%),79427 (10.6%),86128 (10.1%),82438 (8.7%),73614 (7.5%)
CIRUGIA,113890 (10.8%),77292 (10.8%),70743 (9.4%),80385 (9.4%),84002 (8.9%),86304 (8.7%)
MEDICINA,88656 (8.4%),79150 (11.0%),77767 (10.4%),70092 (8.2%),70290 (7.4%),79444 (8.1%)
PEDIATRIA,58659 (5.6%),28154 (3.9%),31390 (4.2%),40884 (4.8%),47695 (5.0%),45102 (4.6%)
...,...,...,...,...,...,...
PSIQUIATRIA FORENSE ALTA COMPLEJIDAD,0 (0.0%),0 (0.0%),0 (0.0%),0 (0.0%),1 (0.0%),9 (0.0%)
CARDIOCIRUGIA (INFANTIL),1 (0.0%),1 (0.0%),1 (0.0%),0 (0.0%),0 (0.0%),0 (0.0%)
PSIQUIATRIA CRONICO,2 (0.0%),1 (0.0%),0 (0.0%),0 (0.0%),0 (0.0%),0 (0.0%)
NULOS,0 (0.0%),1 (0.0%),0 (0.0%),0 (0.0%),0 (0.0%),0 (0.0%)


### **15. Servicios de Traslado (SERVICIOTRASLADO1 al 9)**
- **Tipo de variable**: Hospitalaria / Operativa.
- **Descripción**: Registro cronológico de las unidades o servicios clínicos específicos a los que el paciente fue trasladado durante la evolución de su hospitalización.
- **Cardinalidad**: Muy Alta y variable (entre 66 y 97 categorías por columna, con múltiples errores tipográficos y códigos numéricos sin descripción).
- **Veredicto**: **Descartar columnas originales y Transformar en Variable Derivada (`CANTIDAD_TRASLADOS`)**. El análisis masivo de datos revela que la matriz es insosteniblemente dispersa: el primer traslado (`SERVICIOTRASLADO1`) posee ~80-86% de valores nulos, escalando drásticamente hasta un 99-100% de nulos entre el cuarto y noveno traslado. Aplicar *One-Hot Encoding* a este bloque inyectaría cientos de columnas vacías, diluyendo el poder estadístico de las variables basales. Sin embargo, el "traslado" en sí mismo es un proxy clínico de inestabilidad o complicación. Por ende, las 9 columnas se colapsarán en una única variable numérica discreta que cuente el número total de traslados por episodio, rescatando la varianza útil sin penalizar la dimensionalidad del modelo.

In [18]:
for i in range(1, 10):
    total, oncologico, control = distribucion_categorica_matriz(carpeta, archivos, mapa_columnas,f"SERVICIOTRASLADO{i}")
    mostrar_resultados(total, oncologico, control, f"SERVICIOTRASLADO{i}")


[SERVICIOTRASLADO1] Extrayendo y procesando datos masivos (Una sola pasada)...
✓ Cardinalidad SERVICIOTRASLADO1: 95 categorías únicas
✓ Archivos CSV generados y guardados en las 3 carpetas correspondientes.


'1. Frecuencia para cohorte total (SERVICIOTRASLADO1):'

,2019,2020,2021,2022,2023,2024
NULOS,985068 (85.6%),617598 (79.0%),648637 (79.4%),770865 (82.6%),872748 (84.0%),912553 (84.0%)
MEDICINA,20641 (1.8%),22337 (2.9%),22322 (2.7%),20295 (2.2%),21444 (2.1%),22333 (2.1%)
UNIDAD DE TRATAMIENTO INTERMEDIO (UTI) (INDIFERENCIADO) ADULTO,12892 (1.1%),14283 (1.8%),16501 (2.0%),14503 (1.6%),16936 (1.6%),19159 (1.8%)
CIRUGIA,14748 (1.3%),15504 (2.0%),12834 (1.6%),10809 (1.2%),10874 (1.0%),11723 (1.1%)
PUERPERIO,15827 (1.4%),12655 (1.6%),9243 (1.1%),11503 (1.2%),7729 (0.7%),7217 (0.7%)
...,...,...,...,...,...,...
PSIQUIATRIA FORENSE ALTA COMPLEJIDAD,1 (0.0%),0 (0.0%),0 (0.0%),0 (0.0%),0 (0.0%),5 (0.0%)
HOSPITAL DE DIA MEDICO,0 (0.0%),1 (0.0%),3 (0.0%),0 (0.0%),0 (0.0%),1 (0.0%)
CIRUGIA TORAX,0 (0.0%),2 (0.0%),0 (0.0%),0 (0.0%),0 (0.0%),2 (0.0%)
0,0 (0.0%),0 (0.0%),0 (0.0%),1 (0.0%),1 (0.0%),1 (0.0%)


'2. Frecuencia para cohorte oncológica (SERVICIOTRASLADO1):'

,2019,2020,2021,2022,2023,2024
NULOS,84613 (86.5%),51742 (79.7%),55193 (81.3%),63912 (81.9%),74625 (82.3%),81075 (81.8%)
UNIDAD DE TRATAMIENTO INTERMEDIO (UTI) (INDIFERENCIADO) ADULTO,2528 (2.6%),1973 (3.0%),1658 (2.4%),2381 (3.0%),2903 (3.2%),3322 (3.4%)
MEDICINA,1697 (1.7%),1853 (2.9%),1388 (2.0%),1602 (2.1%),1861 (2.1%),2070 (2.1%)
CIRUGIA,1726 (1.8%),1919 (3.0%),1555 (2.3%),1524 (2.0%),1741 (1.9%),1982 (2.0%)
UNIDAD DE CUIDADOS INTENSIVOS ADULTO,1085 (1.1%),926 (1.4%),984 (1.4%),981 (1.3%),1095 (1.2%),1331 (1.3%)
...,...,...,...,...,...,...
MEDICINA FISICA Y REHABILITACION,0 (0.0%),0 (0.0%),0 (0.0%),0 (0.0%),0 (0.0%),2 (0.0%)
CIRUGIA MAXILO FACIAL,0 (0.0%),0 (0.0%),0 (0.0%),0 (0.0%),0 (0.0%),1 (0.0%)
AREANEONATOLOGIA CUIDADOS INTERMEDIOS,0 (0.0%),0 (0.0%),0 (0.0%),0 (0.0%),0 (0.0%),1 (0.0%)
PSIQUIATRIA CRONICO,1 (0.0%),0 (0.0%),0 (0.0%),0 (0.0%),0 (0.0%),0 (0.0%)


...

,2019,2020,2021,2022,2023,2024
NULOS,1151271 (100.0%),781761 (100.0%),816740 (100.0%),932666 (100.0%),1039408 (100.0%),1085636 (100.0%)
UNIDAD DE TRATAMIENTO INTERMEDIO (UTI) (INDIFERENCIADO) ADULTO,17 (0.0%),17 (0.0%),15 (0.0%),19 (0.0%),10 (0.0%),16 (0.0%)
UNIDAD DE TRATAMIENTO INTERMEDIO PEDIATRIA,13 (0.0%),8 (0.0%),16 (0.0%),19 (0.0%),11 (0.0%),14 (0.0%)
MEDICINA,13 (0.0%),21 (0.0%),11 (0.0%),5 (0.0%),10 (0.0%),15 (0.0%)
UNIDAD DE CUIDADOS INTENSIVOS PEDIATRIA,16 (0.0%),8 (0.0%),13 (0.0%),11 (0.0%),11 (0.0%),11 (0.0%)
...,...,...,...,...,...,...
PSIQUIATRIA CORTA ESTADIA,0 (0.0%),0 (0.0%),0 (0.0%),0 (0.0%),0 (0.0%),1 (0.0%)
UNIDAD DE AGUDOS TRAUMATOLOGIA,1 (0.0%),0 (0.0%),0 (0.0%),0 (0.0%),0 (0.0%),0 (0.0%)
UNIDAD DE AGUDOS MEDICO QUIRURGICOS,0 (0.0%),1 (0.0%),0 (0.0%),0 (0.0%),0 (0.0%),0 (0.0%)
PSIQUIATRIA,0 (0.0%),0 (0.0%),0 (0.0%),0 (0.0%),0 (0.0%),1 (0.0%)


'2. Frecuencia para cohorte oncológica (SERVICIOTRASLADO9):'

,2019,2020,2021,2022,2023,2024
NULOS,97819 (100.0%),64902 (100.0%),67904 (100.0%),78046 (100.0%),90681 (100.0%),99128 (100.0%)
MEDICINA,1 (0.0%),4 (0.0%),2 (0.0%),2 (0.0%),4 (0.0%),3 (0.0%)
ONCOLOGIA,3 (0.0%),1 (0.0%),4 (0.0%),2 (0.0%),0 (0.0%),3 (0.0%)
UNIDAD DE TRATAMIENTO INTERMEDIO (UTI) (INDIFERENCIADO) ADULTO,1 (0.0%),5 (0.0%),1 (0.0%),0 (0.0%),4 (0.0%),1 (0.0%)
AGUDOS MEDICINA,0 (0.0%),1 (0.0%),1 (0.0%),1 (0.0%),5 (0.0%),2 (0.0%)
CIRUGIA,0 (0.0%),1 (0.0%),2 (0.0%),4 (0.0%),2 (0.0%),1 (0.0%)
UNIDAD DE CUIDADOS INTENSIVOS ADULTO,0 (0.0%),2 (0.0%),1 (0.0%),0 (0.0%),4 (0.0%),1 (0.0%)
AREA MEDICO-QUIRURGICO CUIDADOS MEDIOS,0 (0.0%),2 (0.0%),1 (0.0%),2 (0.0%),0 (0.0%),3 (0.0%)
UNIDAD DE TRATAMIENTO INTERMEDIO MEDICINA ADULTO,1 (0.0%),1 (0.0%),1 (0.0%),0 (0.0%),3 (0.0%),1 (0.0%)
UNIDAD DE CUIDADOS INTENSIVOS PEDIATRIA,0 (0.0%),0 (0.0%),2 (0.0%),1 (0.0%),3 (0.0%),0 (0.0%)


'3. Frecuencia para cohorte control (SERVICIOTRASLADO9):'

,2019,2020,2021,2022,2023,2024
NULOS,1053452 (100.0%),716859 (100.0%),748836 (100.0%),854620 (100.0%),948727 (100.0%),986508 (100.0%)
UNIDAD DE TRATAMIENTO INTERMEDIO (UTI) (INDIFERENCIADO) ADULTO,16 (0.0%),12 (0.0%),14 (0.0%),19 (0.0%),6 (0.0%),15 (0.0%)
UNIDAD DE TRATAMIENTO INTERMEDIO PEDIATRIA,11 (0.0%),7 (0.0%),16 (0.0%),18 (0.0%),11 (0.0%),12 (0.0%)
UNIDAD DE CUIDADOS INTENSIVOS PEDIATRIA,16 (0.0%),8 (0.0%),11 (0.0%),10 (0.0%),8 (0.0%),11 (0.0%)
MEDICINA,12 (0.0%),17 (0.0%),9 (0.0%),3 (0.0%),6 (0.0%),12 (0.0%)
...,...,...,...,...,...,...
AREA CUIDADOS INTENSIVOS PEDIATRICOS,0 (0.0%),0 (0.0%),0 (0.0%),0 (0.0%),1 (0.0%),0 (0.0%)
UNIDAD DE AGUDOS MEDICO QUIRURGICOS,0 (0.0%),1 (0.0%),0 (0.0%),0 (0.0%),0 (0.0%),0 (0.0%)
UNIDAD DE AGUDOS TRAUMATOLOGIA,1 (0.0%),0 (0.0%),0 (0.0%),0 (0.0%),0 (0.0%),0 (0.0%)
PSIQUIATRIA,0 (0.0%),0 (0.0%),0 (0.0%),0 (0.0%),0 (0.0%),1 (0.0%)


### **16. Servicio de Alta (SERVICIOALTA)**
- **Tipo de variable**: Hospitalaria / Operativa.
- **Descripción**: Servicio clínico específico desde donde el paciente es egresado al finalizar su trayectoria hospitalaria.
- **Cardinalidad**: Alta (89 categorías únicas, con redundancias semánticas equivalentes al servicio de ingreso).
- **Veredicto**: **Descartar**. Aunque en un estudio de caracterización retrospectiva la temporalidad no representa un riesgo estricto de fuga de datos (*Data Leakage*), retener esta variable resulta redundante y tautológico. El servicio de alta exhibe una alta colinealidad tanto con el `SERVICIOINGRESO` como con la variable objetivo `TIPOALTA` (Mortalidad). Al conservar el servicio basal (Ingreso) y medir la inestabilidad intrahospitalaria mediante la variable derivada `CANTIDAD_TRASLADOS`, el estado estructural del paciente queda perfectamente caracterizado sin necesidad de lidiar con las 89 categorías ruidosas del servicio final.

In [19]:
total, oncologico, control = distribucion_categorica_matriz(carpeta, archivos, mapa_columnas,"SERVICIOALTA")
mostrar_resultados(total, oncologico, control, "SERVICIOALTA")


[SERVICIOALTA] Extrayendo y procesando datos masivos (Una sola pasada)...
✓ Cardinalidad SERVICIOALTA: 89 categorías únicas
✓ Archivos CSV generados y guardados en las 3 carpetas correspondientes.


'1. Frecuencia para cohorte total (SERVICIOALTA):'

,2019,2020,2021,2022,2023,2024
CIRUGIA,141858 (12.3%),102697 (13.1%),93323 (11.4%),102292 (11.0%),107584 (10.3%),109985 (10.1%)
MEDICINA,121845 (10.6%),110431 (14.1%),107134 (13.1%),99776 (10.7%),103103 (9.9%),115096 (10.6%)
UNIDAD DE RECUPERACION DE PABELLONES (CENTRAL Y CMA),104023 (9.0%),50003 (6.4%),63777 (7.8%),100663 (10.8%),129020 (12.4%),147403 (13.6%)
OBSTETRICIA,91501 (7.9%),74212 (9.5%),71878 (8.8%),76691 (8.2%),79049 (7.6%),71136 (6.6%)
PEDIATRIA,65634 (5.7%),31601 (4.0%),34239 (4.2%),45146 (4.8%),52653 (5.1%),50154 (4.6%)
...,...,...,...,...,...,...
PSIQUIATRIA FORENSE MEDIANA COMPLEJIDAD,20 (0.0%),19 (0.0%),14 (0.0%),25 (0.0%),9 (0.0%),14 (0.0%)
PSIQUIATRIA FORENSE ALTA COMPLEJIDAD,0 (0.0%),0 (0.0%),0 (0.0%),0 (0.0%),1 (0.0%),13 (0.0%)
CIRUGIA TORAX,0 (0.0%),1 (0.0%),0 (0.0%),0 (0.0%),0 (0.0%),1 (0.0%)
CARDIOCIRUGIA (INFANTIL),1 (0.0%),0 (0.0%),0 (0.0%),0 (0.0%),0 (0.0%),0 (0.0%)


'2. Frecuencia para cohorte oncológica (SERVICIOALTA):'

,2019,2020,2021,2022,2023,2024
CIRUGIA,18866 (19.3%),14856 (22.9%),14511 (21.4%),15603 (20.0%),16778 (18.5%),16793 (16.9%)
MEDICINA,14469 (14.8%),11997 (18.5%),9792 (14.4%),10940 (14.0%),12953 (14.3%),15491 (15.6%)
ONCOLOGIA,17156 (17.5%),7674 (11.8%),6431 (9.5%),6457 (8.3%),7269 (8.0%),7774 (7.8%)
GINECOLOGIA,6705 (6.9%),5340 (8.2%),6061 (8.9%),6931 (8.9%),7620 (8.4%),7775 (7.8%)
UNIDAD DE RECUPERACION DE PABELLONES (CENTRAL Y CMA),3905 (4.0%),2494 (3.8%),3608 (5.3%),4863 (6.2%),6078 (6.7%),6811 (6.9%)
...,...,...,...,...,...,...
UNIDAD DE CUIDADOS INTENSIVOS NEONATOLOGIA,2 (0.0%),0 (0.0%),3 (0.0%),4 (0.0%),1 (0.0%),1 (0.0%)
NEONATOLOGIA CUNAS,0 (0.0%),2 (0.0%),0 (0.0%),1 (0.0%),1 (0.0%),1 (0.0%)
MEDICINA FISICA Y REHABILITACION,0 (0.0%),0 (0.0%),0 (0.0%),0 (0.0%),0 (0.0%),4 (0.0%)
UNIDAD DE TRATAMIENTO INTERMEDIO QUEMADOS,0 (0.0%),1 (0.0%),0 (0.0%),0 (0.0%),0 (0.0%),1 (0.0%)


'3. Frecuencia para cohorte control (SERVICIOALTA):'

,2019,2020,2021,2022,2023,2024
MEDICINA,107376 (10.2%),98434 (13.7%),97342 (13.0%),88836 (10.4%),90150 (9.5%),99605 (10.1%)
UNIDAD DE RECUPERACION DE PABELLONES (CENTRAL Y CMA),100118 (9.5%),47509 (6.6%),60169 (8.0%),95800 (11.2%),122942 (13.0%),140592 (14.2%)
CIRUGIA,122992 (11.7%),87841 (12.3%),78812 (10.5%),86689 (10.1%),90806 (9.6%),93192 (9.4%)
OBSTETRICIA,91360 (8.7%),74050 (10.3%),71593 (9.6%),76465 (8.9%),78652 (8.3%),70655 (7.2%)
PEDIATRIA,63354 (6.0%),29729 (4.1%),32297 (4.3%),43518 (5.1%),51023 (5.4%),48527 (4.9%)
...,...,...,...,...,...,...
AREA DE TRANSPLANTE INFANTIL,5 (0.0%),9 (0.0%),3 (0.0%),5 (0.0%),2 (0.0%),5 (0.0%)
PSIQUIATRIA FORENSE ALTA COMPLEJIDAD,0 (0.0%),0 (0.0%),0 (0.0%),0 (0.0%),1 (0.0%),13 (0.0%)
CIRUGIA TORAX,0 (0.0%),1 (0.0%),0 (0.0%),0 (0.0%),0 (0.0%),1 (0.0%)
CARDIOCIRUGIA (INFANTIL),1 (0.0%),0 (0.0%),0 (0.0%),0 (0.0%),0 (0.0%),0 (0.0%)


### **17. Datos Neonatales (CONDICIONDEALTANEONATO, SEXORN, RNESTADO)**
- **Tipo de variable**: Clínica / Hospitalaria.
- **Descripción**: Bloque de variables repetidas (hasta 4 para partos múltiples) que registran la supervivencia, sexo y condición fisiológica de un recién nacido al alta.
- **Cardinalidad**: Muy Baja (1 a 13 categorías útiles, pero con más del 99% de los registros agrupados en nulos o ceros).
- **Veredicto**: **Descartar por varianza nula y contexto clínico**. La extracción de frecuencias demuestra que los valores válidos en estas columnas son estadísticamente inexistentes para la población de interés (los nulos alcanzan el 99.9% en la cohorte oncológica en todos los años). Un parto en una paciente con un diagnóstico principal de neoplasia maligna activa es un evento clínico extremadamente atípico (*outlier* severo). Retener estas 12 columnas (4 de condición, 4 de sexo, 4 de estado) inyectaría vectores de ceros absolutos al conjunto de datos, no aportando información discriminativa para las variables de severidad, mortalidad o consumo de recursos en oncología.

In [20]:
for i in range(1, 5):
    total, oncologico, control = distribucion_categorica_matriz(carpeta, archivos, mapa_columnas,f"CONDICIONDEALTANEONATO{i}")
    mostrar_resultados(total, oncologico, control, f"CONDICIONDEALTANEONATO{i}")
    total, oncologico, control = distribucion_categorica_matriz(carpeta, archivos, mapa_columnas,f"SEXORN{i}")
    mostrar_resultados(total, oncologico, control, f"SEXORN{i}")
    total, oncologico, control = distribucion_categorica_matriz(carpeta, archivos, mapa_columnas,f"RN{i}ESTADO")
    mostrar_resultados(total, oncologico, control, f"RN{i}ESTADO")


[CONDICIONDEALTANEONATO1] Extrayendo y procesando datos masivos (Una sola pasada)...
✓ Cardinalidad CONDICIONDEALTANEONATO1: 2 categorías únicas
✓ Archivos CSV generados y guardados en las 3 carpetas correspondientes.


'1. Frecuencia para cohorte total (CONDICIONDEALTANEONATO1):'

,2019,2020,2021,2022,2023,2024
NULOS,1022454 (88.8%),781910 (100.0%),816909 (100.0%),932837 (100.0%),1039587 (100.0%),1085813 (100.0%)
VIVO,128156 (11.1%),1 (0.0%),0 (0.0%),0 (0.0%),0 (0.0%),0 (0.0%)
FALLECIDO,788 (0.1%),0 (0.0%),0 (0.0%),0 (0.0%),0 (0.0%),0 (0.0%)


'2. Frecuencia para cohorte oncológica (CONDICIONDEALTANEONATO1):'

,2019,2020,2021,2022,2023,2024
NULOS,97754 (99.9%),64926 (100.0%),67926 (100.0%),78069 (100.0%),90715 (100.0%),99153 (100.0%)
VIVO,74 (0.1%),0 (0.0%),0 (0.0%),0 (0.0%),0 (0.0%),0 (0.0%)
FALLECIDO,1 (0.0%),0 (0.0%),0 (0.0%),0 (0.0%),0 (0.0%),0 (0.0%)


...

,2019,2020,2021,2022,2023,2024
NULOS,19 (0.0%),495815 (63.4%),816909 (100.0%),932837 (100.0%),1039587 (100.0%),1085813 (100.0%)
0.0,1151379 (100.0%),286095 (36.6%),0 (0.0%),0 (0.0%),0 (0.0%),0 (0.0%)
9.0,0 (0.0%),1 (0.0%),0 (0.0%),0 (0.0%),0 (0.0%),0 (0.0%)


'2. Frecuencia para cohorte oncológica (RN4ESTADO):'

,2019,2020,2021,2022,2023,2024
NULOS,0 (0.0%),40642 (62.6%),67926 (100.0%),78069 (100.0%),90715 (100.0%),99153 (100.0%)
0.0,97829 (100.0%),24284 (37.4%),0 (0.0%),0 (0.0%),0 (0.0%),0 (0.0%)


'3. Frecuencia para cohorte control (RN4ESTADO):'

,2019,2020,2021,2022,2023,2024
NULOS,19 (0.0%),455173 (63.5%),748983 (100.0%),854768 (100.0%),948872 (100.0%),986660 (100.0%)
0.0,1053550 (100.0%),261811 (36.5%),0 (0.0%),0 (0.0%),0 (0.0%),0 (0.0%)
9.0,0 (0.0%),1 (0.0%),0 (0.0%),0 (0.0%),0 (0.0%),0 (0.0%)


### **18. Diagnóstico Principal de Ingreso (DIAGNOSTICO1)**
- **Tipo de variable**: Clínica.
- **Descripción**: Representa la condición clínica principal que motiva el ingreso hospitalario del paciente al episodio asistencial.
- **Cardinalidad**: Muy alta en su forma nativa (9.711 categorías únicas a lo largo del periodo), reducida a 16 macro-categorías en la etapa de clasificación.
- **Veredicto**: **Retener y Agrupar (`CATEGORIA_CANCER`)**. Mantener los códigos CIE-10 nativos generaría una matriz inmanejable y dispersa. La estrategia implementada de agrupar los códigos oncológicos (C00-C97) en 15 macro-categorías topográficas (más el grupo control `SIN_CANCER`) es la solución óptima. Reduce drásticamente la dimensionalidad de 9.711 a 16, capturando perfectamente el motivo basal del episodio para perfilar al paciente, sin diluir el peso de la variable en los algoritmos de selección de características.

### **19. Diagnósticos Secundarios (DIAGNOSTICO2 al DIAGNOSTICO35)**
- **Tipo de variable**: Clínica.
- **Descripción**: Códigos CIE-10 adicionales que documentan comorbilidades preexistentes (ej. Hipertensión I10, Diabetes E11.9), metástasis en otros órganos, o complicaciones clínicas surgidas durante la hospitalización.
- **Cardinalidad**: Extrema y decreciente (inicia en 15.300 categorías únicas en el diagnóstico 2 y desciende hasta 799 en el diagnóstico 35).
- **Veredicto**: **Descartar columnas originales y Sintetizar**. El análisis masivo revela que la tasa de valores nulos escala aceleradamente: roza el 12-18% en el `DIAGNOSTICO2`, supera el 50% en el `DIAGNOSTICO5` y alcanza el 99-100% desde el `DIAGNOSTICO15` en adelante. Aplicar *One-Hot Encoding* a estas 34 columnas inyectaría más de 100.000 nuevas dimensiones al conjunto de datos, compuestas casi en su totalidad por ceros. Esto provocaría el colapso computacional y la pérdida de capacidad de generalización del modelo (maldición de la dimensionalidad). Su valor clínico se rescatará exclusivamente a través de variables derivadas.

---

### **Variables Derivadas de los Diagnósticos (Ingeniería de Características)**

Para capturar la riqueza clínica de las 35 columnas de diagnóstico sin comprometer la estabilidad matemática del conjunto de datos, se generarán las siguientes variables sintetizadas:

### **a) Categoría Oncológica (CATEGORIA_CANCER)**
- **Tipo de variable**: Clínica Derivada (Categórica).
- **Descripción**: Clasificación topográfica del tumor maligno del paciente. Agrupa los códigos CIE-10 nativos (C00-C97) encontrados durante el escaneo de diagnósticos.
- **Cardinalidad**: Baja (15 macro-categorías oncológicas + 1 categoría `SIN_CANCER`).
- **Veredicto**: **Crear e Incluir**. Es la variable estructural del estudio. Transforma los cientos de códigos oncológicos específicos en 15 sistemas anatómicos manejables. Su creación permite segmentar a la población y habilitar a los algoritmos predictivos para evaluar cómo el tipo específico de cáncer (ej. digestivo vs. respiratorio) altera el perfil de riesgo y el consumo de recursos frente al grupo de control.

### **b) Tipo de Diagnóstico Oncológico (TIPO_DIAGNOSTICO_ONCO)**
- **Tipo de variable**: Clínica Derivada (Categórica).
- **Descripción**: Indicador que clasifica la jerarquía del hallazgo oncológico durante el episodio hospitalario.
- **Cardinalidad**: Muy Baja (3 categorías: `PRINCIPAL`, `SECUNDARIO`, `SIN_CANCER`).
- **Veredicto**: **Crear e Incluir**. Esta variable corrige un sesgo metodológico crítico. Diferencia a los pacientes que ingresaron al hospital *por* su cáncer (`PRINCIPAL` - Diagnóstico 1) de aquellos que ingresaron por otro motivo pero que *padecen* un cáncer activo que complica su cuadro (`SECUNDARIO` - Diagnóstico 2 al 35). Esta distinción es vital para que el selector de características pueda aislar el impacto del tumor como comorbilidad agravante.

### **c) Carga Oncológica (CARGA_ONCOLOGICA)**
- **Tipo de variable**: Clínica Derivada (Numérica discreta).
- **Descripción**: Conteo del total de diagnósticos que inician con la letra "C" (neoplasias malignas, C00-C97) a lo largo de las 35 columnas para un mismo paciente.
- **Cardinalidad**: Baja (rango numérico acotado).
- **Veredicto**: **Crear e Incluir**. Actúa como un proxy clínico sólido de la diseminación de la enfermedad (metástasis) o la presencia de múltiples tumores primarios sincronizados. Esta variable permite a los modelos medir numéricamente la extensión del cáncer, factor que impacta directamente en la severidad y el consumo de recursos.

### **d) Carga de Comorbilidades (NUM_COMORBILIDADES)**
- **Tipo de variable**: Clínica Derivada (Numérica discreta).
- **Descripción**: Conteo del total de códigos CIE-10 que NO inician con la letra "C" (patologías no oncológicas) registrados entre los diagnósticos 2 y 35.
- **Cardinalidad**: Baja (rango numérico acotado).
- **Veredicto**: **Crear e Incluir**. En oncología, el desenlace clínico no depende exclusivamente del tumor. Las comorbilidades descompensadas (ej. fallas renales, cardiovasculares o metabólicas) agravan la estadía. Cuantificar este volumen de patologías secundarias proporciona al algoritmo una métrica directa de la fragilidad sistémica basal del paciente.

### **e) Comorbilidad Principal (COMORBILIDAD_PRINCIPAL)**
- **Tipo de variable**: Clínica Derivada (Categórica).
- **Descripción**: Identificación del primer código CIE-10 no oncológico (que no empieza con "C") registrado en los diagnósticos secundarios, representando la patología crónica o complicación más relevante del paciente.
- **Cardinalidad**: Moderada (Top N agrupado).
- **Veredicto**: **Crear y Agrupar**. Para evitar reintroducir el problema de la alta dimensionalidad, se identificará únicamente el *Top 10 o Top 15* de las enfermedades crónicas más prevalentes extraídas de las matrices de frecuencia (como Hipertensión Arterial [I10] o Diabetes Mellitus [E11.9]). El resto de las afecciones minoritarias se colapsarán bajo la macro-categoría `OTRA_COMORBILIDAD`, y los pacientes sin patologías extra se etiquetarán como `SIN_COMORBILIDAD`.

In [21]:
for i in range(1, 36):
    total, oncologico, control = distribucion_categorica_matriz(carpeta, archivos, mapa_columnas,f"DIAGNOSTICO{i}")
    mostrar_resultados(total, oncologico, control, f"DIAGNOSTICO{i}")


[DIAGNOSTICO1] Extrayendo y procesando datos masivos (Una sola pasada)...
✓ Cardinalidad DIAGNOSTICO1: 9711 categorías únicas
✓ Archivos CSV generados y guardados en las 3 carpetas correspondientes.


'1. Frecuencia para cohorte total (DIAGNOSTICO1):'

,2019,2020,2021,2022,2023,2024
H26.9,30377 (2.6%),13783 (1.8%),17794 (2.2%),26595 (2.9%),31612 (3.0%),33390 (3.1%)
K80.2,23796 (2.1%),12115 (1.5%),12476 (1.5%),18997 (2.0%),21126 (2.0%),20717 (1.9%)
U07.1,0 (0.0%),33047 (4.2%),55156 (6.8%),13161 (1.4%),3128 (0.3%),2153 (0.2%)
K35.8,20401 (1.8%),17581 (2.2%),16999 (2.1%),16042 (1.7%),15863 (1.5%),16638 (1.5%)
O80.0,23911 (2.1%),19765 (2.5%),15073 (1.8%),14000 (1.5%),13278 (1.3%),9472 (0.9%)
...,...,...,...,...,...,...
A00.1,0 (0.0%),1 (0.0%),0 (0.0%),0 (0.0%),0 (0.0%),0 (0.0%)
Z97.1,1 (0.0%),0 (0.0%),0 (0.0%),0 (0.0%),0 (0.0%),0 (0.0%)
Z97.2,1 (0.0%),0 (0.0%),0 (0.0%),0 (0.0%),0 (0.0%),0 (0.0%)
Z97.3,1 (0.0%),0 (0.0%),0 (0.0%),0 (0.0%),0 (0.0%),0 (0.0%)


'2. Frecuencia para cohorte oncológica (DIAGNOSTICO1):'

,2019,2020,2021,2022,2023,2024
Z51.1,23238 (23.8%),8163 (12.6%),7050 (10.4%),7171 (9.2%),8061 (8.9%),9262 (9.3%)
C50.9,4228 (4.3%),2788 (4.3%),3224 (4.7%),4348 (5.6%),4753 (5.2%),4660 (4.7%)
C73,1513 (1.5%),1238 (1.9%),1460 (2.1%),1873 (2.4%),2128 (2.3%),2126 (2.1%)
C16.9,1946 (2.0%),1413 (2.2%),1399 (2.1%),1699 (2.2%),1852 (2.0%),1907 (1.9%)
C61,1949 (2.0%),1214 (1.9%),1283 (1.9%),1595 (2.0%),2024 (2.2%),2147 (2.2%)
...,...,...,...,...,...,...
Z71.3,1 (0.0%),0 (0.0%),0 (0.0%),0 (0.0%),0 (0.0%),0 (0.0%)
Z71.9,0 (0.0%),1 (0.0%),0 (0.0%),0 (0.0%),0 (0.0%),0 (0.0%)
A05.8,1 (0.0%),0 (0.0%),0 (0.0%),0 (0.0%),0 (0.0%),0 (0.0%)
A07.1,0 (0.0%),0 (0.0%),0 (0.0%),1 (0.0%),0 (0.0%),0 (0.0%)


...

,2019,2020,2021,2022,2023,2024
NULOS,1151259 (100.0%),781642 (100.0%),816554 (100.0%),932447 (100.0%),1039065 (99.9%),1085096 (99.9%)
Z92.2,6 (0.0%),3 (0.0%),14 (0.0%),11 (0.0%),20 (0.0%),35 (0.0%)
Z92.4,3 (0.0%),3 (0.0%),2 (0.0%),9 (0.0%),9 (0.0%),18 (0.0%)
I10,0 (0.0%),3 (0.0%),7 (0.0%),5 (0.0%),13 (0.0%),15 (0.0%)
E66.9,1 (0.0%),6 (0.0%),9 (0.0%),8 (0.0%),10 (0.0%),5 (0.0%)
...,...,...,...,...,...,...
I74.9,0 (0.0%),0 (0.0%),0 (0.0%),0 (0.0%),1 (0.0%),0 (0.0%)
I74.8,0 (0.0%),0 (0.0%),0 (0.0%),0 (0.0%),1 (0.0%),0 (0.0%)
I74.5,0 (0.0%),0 (0.0%),0 (0.0%),0 (0.0%),0 (0.0%),1 (0.0%)
Z99.8,0 (0.0%),0 (0.0%),0 (0.0%),0 (0.0%),0 (0.0%),1 (0.0%)


'2. Frecuencia para cohorte oncológica (DIAGNOSTICO35):'

,2019,2020,2021,2022,2023,2024
NULOS,97814 (100.0%),64903 (100.0%),67919 (100.0%),78050 (100.0%),90701 (100.0%),99116 (100.0%)
Z87.1,1 (0.0%),1 (0.0%),0 (0.0%),0 (0.0%),0 (0.0%),1 (0.0%)
I10,0 (0.0%),0 (0.0%),1 (0.0%),2 (0.0%),0 (0.0%),0 (0.0%)
Z87.4,0 (0.0%),0 (0.0%),1 (0.0%),0 (0.0%),1 (0.0%),1 (0.0%)
Z92.2,1 (0.0%),0 (0.0%),0 (0.0%),0 (0.0%),0 (0.0%),2 (0.0%)
...,...,...,...,...,...,...
Z92.6,0 (0.0%),0 (0.0%),0 (0.0%),0 (0.0%),0 (0.0%),1 (0.0%)
Z93.5,0 (0.0%),0 (0.0%),0 (0.0%),1 (0.0%),0 (0.0%),0 (0.0%)
Z95.8,0 (0.0%),0 (0.0%),0 (0.0%),0 (0.0%),0 (0.0%),1 (0.0%)
Z96.0,0 (0.0%),0 (0.0%),0 (0.0%),0 (0.0%),1 (0.0%),0 (0.0%)


'3. Frecuencia para cohorte control (DIAGNOSTICO35):'

,2019,2020,2021,2022,2023,2024
NULOS,1053445 (100.0%),716739 (100.0%),748635 (100.0%),854397 (100.0%),948364 (99.9%),985980 (99.9%)
Z92.2,5 (0.0%),3 (0.0%),14 (0.0%),11 (0.0%),20 (0.0%),33 (0.0%)
Z92.4,3 (0.0%),3 (0.0%),2 (0.0%),9 (0.0%),8 (0.0%),18 (0.0%)
I10,0 (0.0%),3 (0.0%),6 (0.0%),3 (0.0%),13 (0.0%),15 (0.0%)
E66.9,1 (0.0%),5 (0.0%),9 (0.0%),8 (0.0%),10 (0.0%),5 (0.0%)
...,...,...,...,...,...,...
I71.4,0 (0.0%),0 (0.0%),0 (0.0%),0 (0.0%),1 (0.0%),0 (0.0%)
I74.5,0 (0.0%),0 (0.0%),0 (0.0%),0 (0.0%),0 (0.0%),1 (0.0%)
D47.3,0 (0.0%),0 (0.0%),0 (0.0%),0 (0.0%),0 (0.0%),1 (0.0%)
D50.9,0 (0.0%),0 (0.0%),0 (0.0%),0 (0.0%),1 (0.0%),0 (0.0%)


### **20. Procedimientos Médicos (PROCEDIMIENTO1 al PROCEDIMIENTO30)**
- **Tipo de variable**: Clínica / Hospitalaria.
- **Descripción**: Bloque de 30 columnas que registran, mediante códigos CIE-9-MC o similares, las intervenciones médicas, quirúrgicas, diagnósticas o terapéuticas realizadas al paciente durante su hospitalización.
- **Cardinalidad**: Extrema y decreciente (inicia en 3.999 categorías únicas en el procedimiento 1, disminuyendo hasta ~760 categorías en el procedimiento 30).
- **Veredicto**: **Descartar columnas originales y Sintetizar**. Dado que el objetivo es caracterizar el consumo de recursos y la severidad del perfil oncológico en retrospectiva, no existe riesgo de fuga de datos. Sin embargo, el análisis de frecuencias masivo revela un obstáculo matemático severo: la variable `PROCEDIMIENTO2` ya presenta una tasa de nulos de entre 6.3% y 11.5%, la cual escala drásticamente hasta alcanzar un 99.3% de nulos en el `PROCEDIMIENTO30`. Aplicar *One-Hot Encoding* a este bloque inyectaría aproximadamente 30.000 nuevas columnas al conjunto de datos, provocando un sobreajuste masivo (*overfitting*) y el colapso de la memoria durante el entrenamiento de los algoritmos. Su valor predictivo se extraerá exclusivamente mediante variables derivadas.

---

### **Variables Derivadas de los Procedimientos (Ingeniería de Características)**

Para aprovechar la información crítica sobre las intervenciones hospitalarias sin sufrir la "maldición de la dimensionalidad", se crearán las siguientes variables consolidadas:

### **a) Carga de Procedimientos (NUM_PROCEDIMIENTOS)**
- **Tipo de variable**: Clínica Derivada (Numérica discreta).
- **Descripción**: Conteo del total de procedimientos válidos (no nulos) registrados a lo largo de las 30 columnas para un mismo paciente durante el episodio.
- **Cardinalidad**: Baja (rango numérico continuo acotado, ej. de 0 a 30).
- **Veredicto**: **Crear e Incluir**. Esta variable actúa como un proxy directo del esfuerzo clínico y del consumo de recursos. Los algoritmos de selección de características (como XGBoost) podrán utilizar este conteo para diferenciar numéricamente los episodios oncológicos altamente invasivos (múltiples cirugías o terapias complejas) de aquellos de manejo conservador.

### **b) Intervención Quirúrgica Principal (PROCEDIMIENTO_QUIRURGICO_PRINCIPAL)**
- **Tipo de variable**: Clínica Derivada (Categórica binarizada o agrupada).
- **Descripción**: Identificación del procedimiento más relevante (usualmente el registrado en `PROCEDIMIENTO1`) o clasificación del paciente según si fue sometido a una intervención quirúrgica mayor vs. tratamiento médico exclusivo.
- **Cardinalidad**: Muy Baja (si se binariza: *Sí/No*; o moderada si se agrupa en *Top 10 procedimientos*).
- **Veredicto**: **Crear e Incluir**. Para evitar las más de 3.900 categorías de la primera columna, se propone mapear el `PROCEDIMIENTO1` (o buscar en todo el bloque) utilizando un diccionario que agrupe los códigos en categorías de alta relevancia oncológica, tales como: *Cirugía de Resección Tumoral*, *Quimioterapia/Radioterapia*, *Biopsia/Diagnóstico Invasivo*, o *Procedimiento de Soporte*. Alternativamente, se puede binarizar simplemente como paciente "Quirúrgico" o "Médico". Esto permitirá al modelo asociar intervenciones invasivas con métricas de severidad o prolongación de la estadía.

In [22]:
for i in range(1, 31):
    total, oncologico, control = distribucion_categorica_matriz(carpeta, archivos, mapa_columnas,f"PROCEDIMIENTO{i}")
    mostrar_resultados(total, oncologico, control, f"PROCEDIMIENTO{i}")


[PROCEDIMIENTO1] Extrayendo y procesando datos masivos (Una sola pasada)...


✓ Cardinalidad PROCEDIMIENTO1: 3999 categorías únicas
✓ Archivos CSV generados y guardados en las 3 carpetas correspondientes.


'1. Frecuencia para cohorte total (PROCEDIMIENTO1):'

,2019,2020,2021,2022,2023,2024
73.59,68095 (5.9%),57658 (7.4%),50072 (6.1%),57559 (6.2%),55205 (5.3%),48546 (4.5%)
74.1,53690 (4.7%),45258 (5.8%),40735 (5.0%),46571 (5.0%),45974 (4.4%),42964 (4.0%)
51.23,41824 (3.6%),27527 (3.5%),29882 (3.7%),37397 (4.0%),42029 (4.0%),43394 (4.0%)
13.41,41636 (3.6%),20170 (2.6%),25214 (3.1%),37005 (4.0%),47898 (4.6%),48619 (4.5%)
87.03,35318 (3.1%),24939 (3.2%),24905 (3.0%),28057 (3.0%),29506 (2.8%),32838 (3.0%)
...,...,...,...,...,...,...
77.72,0 (0.0%),0 (0.0%),0 (0.0%),1 (0.0%),0 (0.0%),0 (0.0%)
0.13,0 (0.0%),1 (0.0%),0 (0.0%),0 (0.0%),0 (0.0%),0 (0.0%)
0.1,0 (0.0%),1 (0.0%),0 (0.0%),0 (0.0%),0 (0.0%),0 (0.0%)
0.47,0 (0.0%),1 (0.0%),0 (0.0%),0 (0.0%),0 (0.0%),0 (0.0%)


'2. Frecuencia para cohorte oncológica (PROCEDIMIENTO1):'

...

,2019,2020,2021,2022,2023,2024
NULOS,1142853 (99.3%),770463 (98.5%),799219 (97.8%),917832 (98.4%),1023576 (98.5%),1069260 (98.5%)
93.18,288 (0.0%),358 (0.0%),534 (0.1%),524 (0.1%),517 (0.0%),575 (0.1%)
90.59,277 (0.0%),290 (0.0%),478 (0.1%),457 (0.0%),527 (0.1%),571 (0.1%)
99.29,180 (0.0%),223 (0.0%),407 (0.0%),378 (0.0%),403 (0.0%),438 (0.0%)
89.65,231 (0.0%),282 (0.0%),464 (0.1%),336 (0.0%),340 (0.0%),366 (0.0%)
...,...,...,...,...,...,...
2.31,0 (0.0%),0 (0.0%),0 (0.0%),0 (0.0%),1 (0.0%),0 (0.0%)
0.5,0 (0.0%),0 (0.0%),0 (0.0%),1 (0.0%),0 (0.0%),0 (0.0%)
0.42,0 (0.0%),1 (0.0%),0 (0.0%),0 (0.0%),0 (0.0%),0 (0.0%)
0.39,0 (0.0%),0 (0.0%),0 (0.0%),0 (0.0%),0 (0.0%),1 (0.0%)


'2. Frecuencia para cohorte oncológica (PROCEDIMIENTO30):'

,2019,2020,2021,2022,2023,2024
NULOS,96720 (98.9%),63727 (98.2%),66341 (97.7%),76237 (97.7%),88511 (97.6%),96818 (97.6%)
93.18,42 (0.0%),41 (0.1%),45 (0.1%),66 (0.1%),77 (0.1%),90 (0.1%)
90.59,34 (0.0%),30 (0.0%),45 (0.1%),54 (0.1%),72 (0.1%),72 (0.1%)
99.29,28 (0.0%),16 (0.0%),47 (0.1%),54 (0.1%),46 (0.1%),61 (0.1%)
57.94,28 (0.0%),34 (0.1%),36 (0.1%),39 (0.0%),45 (0.0%),59 (0.1%)
...,...,...,...,...,...,...
97.88,0 (0.0%),0 (0.0%),0 (0.0%),0 (0.0%),1 (0.0%),0 (0.0%)
0.4,0 (0.0%),0 (0.0%),0 (0.0%),0 (0.0%),1 (0.0%),0 (0.0%)
99.78,0 (0.0%),0 (0.0%),1 (0.0%),0 (0.0%),0 (0.0%),0 (0.0%)
99.83,0 (0.0%),0 (0.0%),0 (0.0%),1 (0.0%),0 (0.0%),0 (0.0%)


'3. Frecuencia para cohorte control (PROCEDIMIENTO30):'

,2019,2020,2021,2022,2023,2024
NULOS,1046133 (99.3%),706736 (98.6%),732878 (97.8%),841595 (98.5%),935065 (98.5%),972442 (98.6%)
93.18,246 (0.0%),317 (0.0%),489 (0.1%),458 (0.1%),440 (0.0%),485 (0.0%)
90.59,243 (0.0%),260 (0.0%),433 (0.1%),403 (0.0%),455 (0.0%),499 (0.1%)
89.65,202 (0.0%),243 (0.0%),419 (0.1%),301 (0.0%),300 (0.0%),316 (0.0%)
99.29,152 (0.0%),207 (0.0%),360 (0.0%),324 (0.0%),357 (0.0%),377 (0.0%)
...,...,...,...,...,...,...
99.48,0 (0.0%),0 (0.0%),0 (0.0%),1 (0.0%),0 (0.0%),0 (0.0%)
99.51,0 (0.0%),0 (0.0%),0 (0.0%),1 (0.0%),0 (0.0%),0 (0.0%)
99.96,0 (0.0%),0 (0.0%),1 (0.0%),0 (0.0%),0 (0.0%),0 (0.0%)
0.09,0 (0.0%),0 (0.0%),1 (0.0%),0 (0.0%),0 (0.0%),0 (0.0%)


### **21. Especialidad de Intervención (ESPECIALIDADINTERVENCION)**
- **Tipo de variable**: Hospitalaria.
- **Descripción**: Especialidad médica o quirúrgica del profesional que realizó una intervención específica durante la hospitalización.
- **Cardinalidad**: Alta (104 categorías únicas).
- **Veredicto**: **Descartar**. Aunque en este diseño de caracterización retrospectiva no hay riesgo de fuga de datos, la variable presenta graves problemas de densidad de la información. Entre el 48% y el 63% de los registros en la cohorte oncológica son nulos, lo cual es lógicamente esperado ya que no todos los pacientes ingresados requieren intervención quirúrgica. Mantener una variable tan fragmentada aportará ruido innecesario, sobre todo considerando que la variable estructural `ESPECIALIDAD_MEDICA` ya será retenida, agrupada y mapeada en el modelo.

In [23]:
total, oncologico, control = distribucion_categorica_matriz(carpeta, archivos, mapa_columnas,"ESPECIALIDADINTERVENCION")
mostrar_resultados(total, oncologico, control, "ESPECIALIDADINTERVENCION")


[ESPECIALIDADINTERVENCION] Extrayendo y procesando datos masivos (Una sola pasada)...
✓ Cardinalidad ESPECIALIDADINTERVENCION: 104 categorías únicas
✓ Archivos CSV generados y guardados en las 3 carpetas correspondientes.


'1. Frecuencia para cohorte total (ESPECIALIDADINTERVENCION):'

,2019,2020,2021,2022,2023,2024
NULOS,543389 (47.2%),369837 (47.3%),384249 (47.0%),398021 (42.7%),440727 (42.4%),465915 (42.9%)
CIRUGIA GENERAL,139715 (12.1%),97290 (12.4%),102871 (12.6%),122801 (13.2%),133431 (12.8%),140966 (13.0%)
OBSTETRICIA Y GINECOLOGIA,126346 (11.0%),103155 (13.2%),105027 (12.9%),118808 (12.7%),120993 (11.6%),121169 (11.2%)
TRAUMATOLOGIA Y ORTOPEDIA,69895 (6.1%),49030 (6.3%),54794 (6.7%),67386 (7.2%),81827 (7.9%),84729 (7.8%)
OFTALMOLOGIA,63131 (5.5%),30351 (3.9%),37268 (4.6%),53023 (5.7%),66180 (6.4%),67594 (6.2%)
...,...,...,...,...,...,...
ADOLESCENCIA (PEDIATRIA),0 (0.0%),0 (0.0%),1 (0.0%),0 (0.0%),0 (0.0%),0 (0.0%)
FACOERESIS EXTRACAPSULAR CON I,1 (0.0%),0 (0.0%),0 (0.0%),0 (0.0%),0 (0.0%),0 (0.0%)
ADOLESCENCIA,0 (0.0%),0 (0.0%),0 (0.0%),1 (0.0%),0 (0.0%),0 (0.0%)
MEDICINA NUCLEAR,0 (0.0%),0 (0.0%),0 (0.0%),0 (0.0%),1 (0.0%),0 (0.0%)


'2. Frecuencia para cohorte oncológica (ESPECIALIDADINTERVENCION):'

,2019,2020,2021,2022,2023,2024
NULOS,62502 (63.9%),36922 (56.9%),34855 (51.3%),38064 (48.8%),44062 (48.6%),49513 (49.9%)
CIRUGIA GENERAL,15860 (16.2%),13033 (20.1%),14824 (21.8%),17435 (22.3%),19344 (21.3%),19616 (19.8%)
UROLOGIA,4701 (4.8%),3703 (5.7%),4397 (6.5%),5156 (6.6%),6146 (6.8%),6667 (6.7%)
OBSTETRICIA Y GINECOLOGIA,4475 (4.6%),3558 (5.5%),4121 (6.1%),5126 (6.6%),5806 (6.4%),5967 (6.0%)
"CIRUGIA DE CABEZA, CUELLO Y MAXILOFACIAL",0 (0.0%),774 (1.2%),1026 (1.5%),1553 (2.0%),2006 (2.2%),2553 (2.6%)
...,...,...,...,...,...,...
ENFERNEDADES RESPIRATORIAS PEDIATRICA,1 (0.0%),0 (0.0%),0 (0.0%),0 (0.0%),0 (0.0%),0 (0.0%)
GENETICA CLINICA,1 (0.0%),0 (0.0%),0 (0.0%),0 (0.0%),0 (0.0%),0 (0.0%)
GERIATRIA,0 (0.0%),0 (0.0%),0 (0.0%),0 (0.0%),0 (0.0%),1 (0.0%)
NEFROLOGIA PEDIATRICA,1 (0.0%),0 (0.0%),0 (0.0%),0 (0.0%),0 (0.0%),0 (0.0%)


'3. Frecuencia para cohorte control (ESPECIALIDADINTERVENCION):'

,2019,2020,2021,2022,2023,2024
NULOS,480887 (45.6%),332915 (46.4%),349394 (46.6%),359957 (42.1%),396665 (41.8%),416402 (42.2%)
OBSTETRICIA Y GINECOLOGIA,121871 (11.6%),99597 (13.9%),100906 (13.5%),113682 (13.3%),115187 (12.1%),115202 (11.7%)
CIRUGIA GENERAL,123855 (11.8%),84257 (11.8%),88047 (11.8%),105366 (12.3%),114087 (12.0%),121350 (12.3%)
TRAUMATOLOGIA Y ORTOPEDIA,69024 (6.6%),48283 (6.7%),53863 (7.2%),66273 (7.8%),80315 (8.5%),83069 (8.4%)
OFTALMOLOGIA,62426 (5.9%),30002 (4.2%),36732 (4.9%),52236 (6.1%),65186 (6.9%),66521 (6.7%)
...,...,...,...,...,...,...
ADOLESCENCIA (PEDIATRIA),0 (0.0%),0 (0.0%),1 (0.0%),0 (0.0%),0 (0.0%),0 (0.0%)
FACOERESIS EXTRACAPSULAR CON I,1 (0.0%),0 (0.0%),0 (0.0%),0 (0.0%),0 (0.0%),0 (0.0%)
ADOLESCENCIA,0 (0.0%),0 (0.0%),0 (0.0%),1 (0.0%),0 (0.0%),0 (0.0%)
MEDICINA NUCLEAR,0 (0.0%),0 (0.0%),0 (0.0%),0 (0.0%),1 (0.0%),0 (0.0%)


### **22. Uso de Pabellón (USOSPABELLON)**
- **Tipo de variable**: Hospitalaria.
- **Descripción**: Indica si el paciente utilizó un quirófano (pabellón) y de qué tipo (central, urgencia, ambulatorio, etc.).
- **Cardinalidad**: Alta y ruidosa (71 categorías únicas, superando ampliamente las 8 esperadas según el diccionario oficial).
- **Veredicto**: **Retener, Limpiar y Binarizar**. En el contexto de un modelo explicativo (donde se descartó el problema de fuga temporal de datos), el uso de pabellón es un proxy fundamental del nivel de invasividad y consumo de recursos del episodio oncológico. No obstante, la columna está contaminada con errores de digitación (ej. códigos alfanuméricos como "7001546-3" o "A?AO") y presenta cerca de un 45-60% de valores nulos. Para evitar dispersión, se transformará en una variable dicotómica `USO_PABELLON` (1 = Sí, 0 = No), colapsando los códigos válidos en la clase positiva y asumiendo los nulos y valores anómalos como ausencia de uso quirúrgico.
##### **Tabla (diccionario de datos extraído del sitio de "Fonasa - Datos abiertos")**:

| ID | Tipo de pabellón                         |
|----|------------------------------------------|
| 1  | Pabellón Central u Obstétrico           |
| 2  | Pabellón ambulatorio                    |
| 3  | Sala de procedimiento                   |
| 4  | Pabellón Hemodinamia                    |
| 5  | Pabellón de Urgencia                    |
| 6  | Pabellón de Urgencia habilitado en UCI  |
| 7  | H. de Campaña                           |
| 8  | Operativos Especiales                   |

## Evaluación y Descarte de Variable Operativa: Uso de Pabellón (USOPABELLON)

### 1. Descripción de la Variable
La variable `USOPABELLON` (teóricamente codificada del 1 al 8 según el estándar MINSAL) buscaba registrar el tipo de quirófano utilizado durante el episodio clínico (ej. Pabellón Central, Ambulatorio, Hemodinamia). 

### 2. Análisis de Calidad de Datos (CRISP-DM: Data Understanding)
Al auditar las frecuencias transversales de la variable para el sexenio 2019-2024, se detectaron fallas estructurales insalvables que inhabilitan su uso en cualquier modelo predictivo:

* **Tasa de Valores Nulos Crítica y Sesgo Temporal:** * Históricamente, la variable presenta un ~40% de datos faltantes (2021-2024).
  * **Falla Catastrófica en 2020:** La variable registra un **100.0% de valores nulos para todo el año 2020**. Esto impide cualquier análisis longitudinal y descarta la hipótesis de que un "nulo" significa "no se usó pabellón" (ya que la actividad quirúrgica, aunque reducida por la pandemia, nunca fue cero).
* **Corrupción Estructural de la Base de Datos:** Los registros de 2019 evidencian un problema de *data leakage* o desplazamiento de columnas desde el sistema origen (MINSAL). La columna contiene RUTs de pacientes (ej. `15851762-0`), fechas (`15/07/2019`) y caracteres corruptos (`A?AO`), demostrando que el campo almacenó basura informática en lugar del código esperado.

### 3. Redundancia de la Información
La pérdida de esta columna no afecta el poder predictivo del modelo, ya que la señal de "complejidad quirúrgica" ya fue capturada exitosamente y con alta calidad a través de tres variables ya procesadas:
1. `TIPO_ACTIVIDAD` (Aísla los casos de Cirugía Mayor Ambulatoria - CMA).
2. `SERVICIOINGRESO` (Aísla los ingresos al Área Quirúrgica).
3. `PROCEDIMIENTO1` (Define anatómicamente la cirugía realizada).

### 4. Veredicto Final
**Variable Descartada.** Se elimina la columna `USOPABELLON` del dataset definitivo por violación de los principios de integridad de datos, exceso de nulos (>40% a 100%) y corrupción de formato. Imputar estos valores introduciría un sesgo inaceptable en el modelo.


In [24]:
total, oncologico, control = distribucion_categorica_matriz(carpeta, archivos, mapa_columnas,"USOSPABELLON")
mostrar_resultados(total, oncologico, control, "USOSPABELLON")


[USOSPABELLON] Extrayendo y procesando datos masivos (Una sola pasada)...
✓ Cardinalidad USOSPABELLON: 71 categorías únicas
✓ Archivos CSV generados y guardados en las 3 carpetas correspondientes.


'1. Frecuencia para cohorte total (USOSPABELLON):'

,2019,2020,2021,2022,2023,2024
NULOS,509092 (44.2%),781857 (100.0%),354166 (43.4%),367631 (39.4%),404206 (38.9%),430231 (39.6%)
1.0,0 (0.0%),40 (0.0%),378419 (46.3%),454467 (48.7%),495894 (47.7%),503243 (46.3%)
1,534536 (46.4%),0 (0.0%),0 (0.0%),0 (0.0%),0 (0.0%),0 (0.0%)
2.0,0 (0.0%),4 (0.0%),47869 (5.9%),68339 (7.3%),87929 (8.5%),91381 (8.4%)
3.0,0 (0.0%),10 (0.0%),24166 (3.0%),25689 (2.8%),30293 (2.9%),34844 (3.2%)
...,...,...,...,...,...,...
7001546-3,1 (0.0%),0 (0.0%),0 (0.0%),0 (0.0%),0 (0.0%),0 (0.0%)
7108401-9,1 (0.0%),0 (0.0%),0 (0.0%),0 (0.0%),0 (0.0%),0 (0.0%)
741,1 (0.0%),0 (0.0%),0 (0.0%),0 (0.0%),0 (0.0%),0 (0.0%)
A?AO,1 (0.0%),0 (0.0%),0 (0.0%),0 (0.0%),0 (0.0%),0 (0.0%)


'2. Frecuencia para cohorte oncológica (USOSPABELLON):'

,2019,2020,2021,2022,2023,2024
NULOS,59059 (60.4%),64919 (100.0%),31997 (47.1%),35444 (45.4%),40570 (44.7%),45666 (46.1%)
1.0,0 (0.0%),3 (0.0%),29865 (44.0%),35143 (45.0%),39776 (43.8%),41629 (42.0%)
1,33486 (34.2%),0 (0.0%),0 (0.0%),0 (0.0%),0 (0.0%),0 (0.0%)
2.0,0 (0.0%),1 (0.0%),3377 (5.0%),4190 (5.4%),5936 (6.5%),6097 (6.1%)
3.0,0 (0.0%),3 (0.0%),2283 (3.4%),2677 (3.4%),3579 (3.9%),4445 (4.5%)
2,2996 (3.1%),0 (0.0%),0 (0.0%),0 (0.0%),0 (0.0%),0 (0.0%)
3,2244 (2.3%),0 (0.0%),0 (0.0%),0 (0.0%),0 (0.0%),0 (0.0%)
4.0,0 (0.0%),0 (0.0%),250 (0.4%),409 (0.5%),562 (0.6%),714 (0.7%)
5.0,0 (0.0%),0 (0.0%),148 (0.2%),194 (0.2%),257 (0.3%),364 (0.4%)
6.0,0 (0.0%),0 (0.0%),2 (0.0%),12 (0.0%),28 (0.0%),201 (0.2%)


'3. Frecuencia para cohorte control (USOSPABELLON):'

,2019,2020,2021,2022,2023,2024
NULOS,450033 (42.7%),716938 (100.0%),322169 (43.0%),332187 (38.9%),363636 (38.3%),384565 (39.0%)
1.0,0 (0.0%),37 (0.0%),348554 (46.5%),419324 (49.1%),456118 (48.1%),461614 (46.8%)
1,501050 (47.6%),0 (0.0%),0 (0.0%),0 (0.0%),0 (0.0%),0 (0.0%)
2.0,0 (0.0%),3 (0.0%),44492 (5.9%),64149 (7.5%),81993 (8.6%),85284 (8.6%)
3.0,0 (0.0%),7 (0.0%),21883 (2.9%),23012 (2.7%),26714 (2.8%),30399 (3.1%)
...,...,...,...,...,...,...
7001546-3,1 (0.0%),0 (0.0%),0 (0.0%),0 (0.0%),0 (0.0%),0 (0.0%)
7108401-9,1 (0.0%),0 (0.0%),0 (0.0%),0 (0.0%),0 (0.0%),0 (0.0%)
741,1 (0.0%),0 (0.0%),0 (0.0%),0 (0.0%),0 (0.0%),0 (0.0%)
A?AO,1 (0.0%),0 (0.0%),0 (0.0%),0 (0.0%),0 (0.0%),0 (0.0%)


### **23. Código GRD (IR_29301_COD_GRD)**
- **Tipo de variable**: Hospitalaria / Administrativa.
- **Descripción**: Código final del Grupo Relacionado por el Diagnóstico asignado por el agrupador de FONASA al egreso.
- **Cardinalidad**: Muy Alta (1.671 categorías únicas extraídas).
- **Veredicto**: **Descartar**. Este código es el resultado determinista del algoritmo de facturación, calculado *a posteriori* combinando diagnósticos, procedimientos y edad. Dado que los modelos de aprendizaje automático buscarán caracterizar la severidad y el consumo de recursos (cuyo peso ponderado se extrae directamente de este mismo código), incluirlo generaría una colinealidad perfecta o "circularidad matemática". Sumado a ello, aplicar *One-Hot Encoding* sobre 1.671 categorías colapsaría la capacidad de convergencia y generalización de los selectores de características.

In [25]:
total, oncologico, control = distribucion_categorica_matriz(carpeta, archivos, mapa_columnas,"IR_29301_COD_GRD")
mostrar_resultados(total, oncologico, control, "IR_29301_COD_GRD")


[IR_29301_COD_GRD] Extrayendo y procesando datos masivos (Una sola pasada)...
✓ Cardinalidad IR_29301_COD_GRD: 1671 categorías únicas
✓ Archivos CSV generados y guardados en las 3 carpetas correspondientes.


'1. Frecuencia para cohorte total (IR_29301_COD_GRD):'

,2019,2020,2021,2022,2023,2024
022360,42984 (3.7%),0 (0.0%),0 (0.0%),39235 (4.2%),50288 (4.8%),51237 (4.7%)
146101,38970 (3.4%),31185 (4.0%),26192 (3.2%),29000 (3.1%),28342 (2.7%),26664 (2.5%)
146121,26649 (2.3%),22253 (2.8%),19667 (2.4%),22720 (2.4%),23269 (2.2%),19729 (1.8%)
146131,30970 (2.7%),25076 (3.2%),19920 (2.4%),21512 (2.3%),19274 (1.9%),17042 (1.6%)
134161,19176 (1.7%),14583 (1.9%),13550 (1.7%),14214 (1.5%),14927 (1.4%),14689 (1.4%)
...,...,...,...,...,...,...
12110,0 (0.0%),0 (0.0%),1 (0.0%),0 (0.0%),0 (0.0%),0 (0.0%)
159160,1 (0.0%),0 (0.0%),0 (0.0%),0 (0.0%),0 (0.0%),0 (0.0%)
23130,0 (0.0%),1 (0.0%),0 (0.0%),0 (0.0%),0 (0.0%),0 (0.0%)
51011,0 (0.0%),1 (0.0%),0 (0.0%),0 (0.0%),0 (0.0%),0 (0.0%)


'2. Frecuencia para cohorte oncológica (IR_29301_COD_GRD):'

,2019,2020,2021,2022,2023,2024
174132,6076 (6.2%),4650 (7.2%),3950 (5.8%),4104 (5.3%),4375 (4.8%),5020 (5.1%)
174133,2620 (2.7%),2310 (3.6%),2163 (3.2%),2030 (2.6%),2453 (2.7%),2566 (2.6%)
174131,3178 (3.2%),2058 (3.2%),1607 (2.4%),1424 (1.8%),1933 (2.1%),2141 (2.2%)
161201,2156 (2.2%),1733 (2.7%),1794 (2.6%),2140 (2.7%),2190 (2.4%),2309 (2.3%)
092420,2137 (2.2%),0 (0.0%),0 (0.0%),2427 (3.1%),3615 (4.0%),3596 (3.6%)
...,...,...,...,...,...,...
051021,0 (0.0%),0 (0.0%),0 (0.0%),0 (0.0%),0 (0.0%),1 (0.0%)
224103,0 (0.0%),0 (0.0%),1 (0.0%),0 (0.0%),0 (0.0%),0 (0.0%)
11011,0 (0.0%),1 (0.0%),0 (0.0%),0 (0.0%),0 (0.0%),0 (0.0%)
111011,0 (0.0%),0 (0.0%),0 (0.0%),1 (0.0%),0 (0.0%),0 (0.0%)


'3. Frecuencia para cohorte control (IR_29301_COD_GRD):'

,2019,2020,2021,2022,2023,2024
022360,42646 (4.0%),0 (0.0%),0 (0.0%),38752 (4.5%),49711 (5.2%),50630 (5.1%)
146101,38951 (3.7%),31158 (4.3%),26168 (3.5%),28974 (3.4%),28313 (3.0%),26636 (2.7%)
146121,26642 (2.5%),22244 (3.1%),19656 (2.6%),22715 (2.7%),23263 (2.5%),19720 (2.0%)
146131,30961 (2.9%),25072 (3.5%),19916 (2.7%),21506 (2.5%),19270 (2.0%),17032 (1.7%)
134161,19164 (1.8%),14571 (2.0%),13538 (1.8%),14194 (1.7%),14916 (1.6%),14672 (1.5%)
...,...,...,...,...,...,...
012110,0 (0.0%),0 (0.0%),0 (0.0%),0 (0.0%),1 (0.0%),0 (0.0%)
173100,1 (0.0%),0 (0.0%),0 (0.0%),0 (0.0%),0 (0.0%),0 (0.0%)
51011,0 (0.0%),1 (0.0%),0 (0.0%),0 (0.0%),0 (0.0%),0 (0.0%)
990139,0 (0.0%),1 (0.0%),0 (0.0%),0 (0.0%),0 (0.0%),0 (0.0%)


### **24. Probabilidad de Mortalidad GRD (IR_29301_MORTALIDAD)**
- **Tipo de variable**: Clínica / Calculada.
- **Descripción**: Probabilidad teórica o nivel de riesgo de mortalidad asignado por el software agrupador GRD al finalizar la estadía.
- **Cardinalidad**: Baja (4 categorías).
- **Veredicto**: **Descartar**. Se trata de un índice sintético pre-calculado por un sistema externo. Retenerlo es redundante y contraproducente metodológicamente: el conjunto de datos ya cuenta con la variable objetivo real y definitiva del desenlace del paciente (`TIPOALTA` binarizada en Vivo/Fallecido). En un modelo de *Machine Learning* explicativo, el objetivo es que el propio algoritmo aprenda a ponderar las reglas y factores de riesgo desde las condiciones clínicas basales y operativas, no obligarlo a "leer" la predicción elaborada por un tercero.

##### **Tabla (diccionario de datos extraído del sitio de "Fonasa - Datos abiertos")**:

| ID | Gravedad     |
|----|--------------|
| 0  | Sin gravedad |
| 1  | Menor        |
| 2  | Moderada     |
| 3  | Mayor        |

In [26]:
total, oncologico, control = distribucion_categorica_matriz(carpeta, archivos, mapa_columnas,"IR_29301_MORTALIDAD")
mostrar_resultados(total, oncologico, control, "IR_29301_MORTALIDAD")


[IR_29301_MORTALIDAD] Extrayendo y procesando datos masivos (Una sola pasada)...
✓ Cardinalidad IR_29301_MORTALIDAD: 4 categorías únicas
✓ Archivos CSV generados y guardados en las 3 carpetas correspondientes.


'1. Frecuencia para cohorte total (IR_29301_MORTALIDAD):'

,2019,2020,2021,2022,2023,2024
1,658994 (57.2%),443943 (56.8%),424123 (51.9%),476812 (51.1%),510177 (49.1%),509445 (46.9%)
0,202517 (17.6%),79664 (10.2%),102938 (12.6%),154772 (16.6%),193550 (18.6%),212051 (19.5%)
2,165823 (14.4%),137766 (17.6%),137256 (16.8%),155117 (16.6%),166014 (16.0%),180092 (16.6%)
3,124043 (10.8%),120538 (15.4%),152592 (18.7%),146113 (15.7%),169816 (16.3%),184210 (17.0%)
DESCONOCIDO,21 (0.0%),0 (0.0%),0 (0.0%),23 (0.0%),30 (0.0%),15 (0.0%)


'2. Frecuencia para cohorte oncológica (IR_29301_MORTALIDAD):'

,2019,2020,2021,2022,2023,2024
2,30206 (30.9%),23050 (35.5%),22595 (33.3%),25523 (32.7%),29591 (32.6%),32344 (32.6%)
3,24186 (24.7%),20997 (32.3%),21803 (32.1%),24983 (32.0%),29706 (32.7%),33896 (34.2%)
1,22762 (23.3%),17231 (26.5%),17839 (26.3%),19652 (25.2%),21203 (23.4%),21830 (22.0%)
0,20675 (21.1%),3648 (5.6%),5689 (8.4%),7909 (10.1%),10215 (11.3%),11081 (11.2%)
DESCONOCIDO,0 (0.0%),0 (0.0%),0 (0.0%),2 (0.0%),0 (0.0%),2 (0.0%)


'3. Frecuencia para cohorte control (IR_29301_MORTALIDAD):'

,2019,2020,2021,2022,2023,2024
1,636232 (60.4%),426712 (59.5%),406284 (54.2%),457160 (53.5%),488974 (51.5%),487615 (49.4%)
0,181842 (17.3%),76016 (10.6%),97249 (13.0%),146863 (17.2%),183335 (19.3%),200970 (20.4%)
2,135617 (12.9%),114716 (16.0%),114661 (15.3%),129594 (15.2%),136423 (14.4%),147748 (15.0%)
3,99857 (9.5%),99541 (13.9%),130789 (17.5%),121130 (14.2%),140110 (14.8%),150314 (15.2%)
DESCONOCIDO,21 (0.0%),0 (0.0%),0 (0.0%),21 (0.0%),30 (0.0%),13 (0.0%)


### **25. Hospital de Procedencia (HOSPPROCEDENCIA)**
- **Tipo de variable**: Hospitalaria.
- **Descripción**: Nombre o código del hospital específico desde el cual el paciente fue derivado antes de ingresar.
- **Cardinalidad**: Muy Alta (518 categorías únicas).
- **Veredicto**: **Descartar por densidad insuficiente**. La extracción de frecuencias revela un patrón categórico insostenible: entre el 89% y el 92% de los episodios oncológicos presentan valores nulos en esta variable. Esto indica empíricamente que 9 de cada 10 pacientes ingresan de forma directa, o bien la derivación inter-hospitalaria específica no se registra de manera sistemática en la red. Inyectar 518 columnas *dummy* para explicar un fenómeno que solo ocurre en menos del 10% de la muestra penalizaría al modelo sin aportar valor generalizable. El impacto epidemiológico de las derivaciones ya está asegurado y sintetizado metodológicamente a través de la variable `TIPO_PROCEDENCIA` macro (ej. categoría agrupada "OTROS HOSPITALES DE LA RED").

In [27]:
total, oncologico, control = distribucion_categorica_matriz(carpeta, archivos, mapa_columnas,"HOSPPROCEDENCIA")
mostrar_resultados(total, oncologico, control, "HOSPPROCEDENCIA")


[HOSPPROCEDENCIA] Extrayendo y procesando datos masivos (Una sola pasada)...
✓ Cardinalidad HOSPPROCEDENCIA: 518 categorías únicas
✓ Archivos CSV generados y guardados en las 3 carpetas correspondientes.


'1. Frecuencia para cohorte total (HOSPPROCEDENCIA):'

,2019,2020,2021,2022,2023,2024
NULOS,1025386 (89.1%),676219 (86.5%),699687 (85.7%),810634 (86.9%),899461 (86.5%),993216 (91.5%)
HOSPITAL SAN PABLO (COQUIMBO),18258 (1.6%),12731 (1.6%),14180 (1.7%),15439 (1.7%),17726 (1.7%),17183 (1.6%)
HOSPITAL CLINICO DE MAGALLANES DR. LAUTARO NAVARRO AVARIA,12059 (1.0%),7859 (1.0%),9046 (1.1%),10354 (1.1%),12018 (1.2%),0 (0.0%)
HOSPITAL SAN JOSE (CORONEL),7557 (0.7%),5617 (0.7%),5743 (0.7%),7349 (0.8%),8537 (0.8%),8975 (0.8%)
HOSPITAL CARLOS VAN BUREN (VALPARAISO),5505 (0.5%),2645 (0.3%),2411 (0.3%),2332 (0.2%),2802 (0.3%),3232 (0.3%)
...,...,...,...,...,...,...
CENTRO DE ATENCION INSTITUTO DE SEGURIDAD DEL TRABAJADOR PUNTA ARENAS,0 (0.0%),0 (0.0%),0 (0.0%),0 (0.0%),1 (0.0%),0 (0.0%)
200417,0 (0.0%),0 (0.0%),1 (0.0%),0 (0.0%),0 (0.0%),0 (0.0%)
101238,0 (0.0%),0 (0.0%),1 (0.0%),0 (0.0%),0 (0.0%),0 (0.0%)
SERVICIOS CLINICOS NEUROPSIQUIATRICOS Y GERIATRICOS R Y G LIMITADA,0 (0.0%),0 (0.0%),1 (0.0%),0 (0.0%),0 (0.0%),0 (0.0%)


'2. Frecuencia para cohorte oncológica (HOSPPROCEDENCIA):'

,2019,2020,2021,2022,2023,2024
NULOS,89800 (91.8%),58540 (90.2%),61016 (89.8%),70415 (90.2%),81256 (89.6%),91929 (92.7%)
HOSPITAL SAN PABLO (COQUIMBO),1163 (1.2%),925 (1.4%),1129 (1.7%),1276 (1.6%),1534 (1.7%),1512 (1.5%)
HOSPITAL CLINICO DE MAGALLANES DR. LAUTARO NAVARRO AVARIA,877 (0.9%),534 (0.8%),627 (0.9%),804 (1.0%),943 (1.0%),0 (0.0%)
HOSPITAL CARLOS VAN BUREN (VALPARAISO),1476 (1.5%),361 (0.6%),327 (0.5%),328 (0.4%),457 (0.5%),504 (0.5%)
HOSPITAL SAN JOSE (CORONEL),326 (0.3%),283 (0.4%),210 (0.3%),241 (0.3%),356 (0.4%),453 (0.5%)
...,...,...,...,...,...,...
CLINICA AMBULATORIA SANATORIO ALEMAN SAN PEDRO,0 (0.0%),0 (0.0%),0 (0.0%),0 (0.0%),1 (0.0%),0 (0.0%)
INTERMEDICA DIVISION DIALISIS LTDA. SUCURSAL LONGAVI,0 (0.0%),0 (0.0%),0 (0.0%),0 (0.0%),1 (0.0%),0 (0.0%)
ONG CORPORACION DE DIALIZADOS Y TRANSPLANTADOS MAYMURU DE ARICA,0 (0.0%),0 (0.0%),0 (0.0%),0 (0.0%),1 (0.0%),0 (0.0%)
NEPHROCARE CHILE S.A. SUCURSAL TALCA,0 (0.0%),0 (0.0%),0 (0.0%),0 (0.0%),1 (0.0%),0 (0.0%)


'3. Frecuencia para cohorte control (HOSPPROCEDENCIA):'

,2019,2020,2021,2022,2023,2024
NULOS,935586 (88.8%),617679 (86.1%),638671 (85.3%),740219 (86.6%),818205 (86.2%),901287 (91.3%)
HOSPITAL SAN PABLO (COQUIMBO),17095 (1.6%),11806 (1.6%),13051 (1.7%),14163 (1.7%),16192 (1.7%),15671 (1.6%)
HOSPITAL CLINICO DE MAGALLANES DR. LAUTARO NAVARRO AVARIA,11182 (1.1%),7325 (1.0%),8419 (1.1%),9550 (1.1%),11075 (1.2%),0 (0.0%)
HOSPITAL SAN JOSE (CORONEL),7231 (0.7%),5334 (0.7%),5533 (0.7%),7108 (0.8%),8181 (0.9%),8522 (0.9%)
HOSPITAL CARLOS VAN BUREN (VALPARAISO),4029 (0.4%),2284 (0.3%),2084 (0.3%),2004 (0.2%),2345 (0.2%),2728 (0.3%)
...,...,...,...,...,...,...
CEMED LIMACHE,0 (0.0%),0 (0.0%),0 (0.0%),0 (0.0%),1 (0.0%),0 (0.0%)
CENTRO DE DIALISIS CLINIDIAL LA SERENA,0 (0.0%),0 (0.0%),0 (0.0%),0 (0.0%),1 (0.0%),0 (0.0%)
101238,0 (0.0%),0 (0.0%),1 (0.0%),0 (0.0%),0 (0.0%),0 (0.0%)
SERVICIOS CLINICOS NEUROPSIQUIATRICOS Y GERIATRICOS R Y G LIMITADA,0 (0.0%),0 (0.0%),1 (0.0%),0 (0.0%),0 (0.0%),0 (0.0%)
